In [0]:
* ================================================================
* Chapter 3 Analysis: School Culture and Pupil Progress
* 3-stage OLS on 102 gold-standard (visit + interview) schools
* Outcome: P8 component 2-year averages (primary) + robustness
* HC3 robust standard errors throughout
* ================================================================

* Ensure user ado packages (esttab/estout) are on the search path
adopath ++ "C:\Users\damia\ado\plus"

* Confirm esttab is findable
which esttab

  [1]              "C:\Users\damia\ado\plus"
  [2]  (BASE)      "C:\Program Files\StataNow19/ado\base/"
  [3]  (SITE)      "C:\Program Files\StataNow19/ado\site/"
  [4]              "."
  [5]  (PERSONAL)  "C:\Users\damia\ado\personal/"
  [6]  (PLUS)      "C:\Users\damia\ado\plus/"
  [7]  (OLDPLACE)  "c:\ado/"
C:\Users\damia\ado\plus\e\esttab.ado
*! version 2.1.4  13apr2026  Ben Jann
*! wrapper for estout


In [1]:
* ---- 1. Load data ----
import delimited "C:/Users/damia/OneDrive/Documents/Schools Project/analysis_dataset.csv", ///
    clear stringcols(_all) case(lower)

* Strip % sign from eal and sen (stored as "57.70%" strings)
foreach v of varlist eal sen {
    replace `v' = subinstr(`v', "%", "", .)
    destring `v', replace
}

* Destring all numeric analysis columns
local numvars gs_warmth_visit gs_strictness_visit gs_teaching_visit ///
    gs_warmth_composite gs_strictness_composite gs_teaching_composite ///
    gs_warmth_score_v1 gs_strictness_score_v1 ///
    gs_w1 gs_w2 gs_w3_adj gs_s1 gs_s2 gs_s3 gs_s4 gs_t1 gs_t2 ///
    p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg ///
    p8mea_2324 p8meaeng_2324 p8meamat_2324 p8meaebac_2324 p8meaopen_2324 ///
    att8screng_2425 att8scrmat_2425 att8screbac_2425 att8scropen_2425 ///
    ks2 fsm log_size academy urban_bin selective ///
    years_since_ofsted ofsted_grade_2019 ///
    ofsted_llmstrictnessscore ///
    trx_llmwarmthscore trx_llmstrictnessscore trx_llmmanagementscore ///
    semh_baseline_2016 semh_current size
destring `numvars', replace force

* Sample flags
gen tier1 = !missing(gs_warmth_visit)      // 102 visited schools
gen tier2 = !missing(gs_warmth_composite)  // 303 interview schools

count if tier1
count if tier2

(encoding automatically selected: UTF-8)


(95 vars, 3,332 obs)
(3,268 real changes made)


eal: all characters numeric; replaced as double
(64 missing values generated)


(3,268 real changes made)


sen: all characters numeric; replaced as double
(64 missing values generated)
gs_warmth_visit: all characters numeric; replaced as double
(3230 missing values generated)


...

ofsted_grade_2019: all characters numeric; replaced as byte
(466 missing values generated)
ofsted_llmstrictnessscore: all characters numeric; replaced as byte
(55 missing values generated)


trx_llmwarmthscore: all characters numeric; replaced as byte
(3042 missing values generated)


trx_llmstrictnessscore: all characters numeric; replaced as byte
(3042 missing values generated)
trx_llmmanagementscore: all characters numeric; replaced as byte
(3042 missing values generated)


semh_baseline_2016: contains nonnumeric characters; replaced as int
(325 missing values generated)
semh_current: all characters numeric; replaced as int


size: all characters numeric; replaced as int
(6 missing values generated)
  102
  303


In [2]:
* ---- 2. Define global macros ----

* Base controls (continuous)
global ctrl_cont "ks2 fsm eal sen log_size years_since_ofsted"

* Binary school-type controls
global ctrl_bin "academy urban_bin selective"

* Pre-COVID Ofsted grade dummies (base = Outstanding, grade 1)
* Missing grade (7 schools) are dropped from regressions with this macro
global ctrl_ofsted "2.ofsted_grade_2019 3.ofsted_grade_2019 4.ofsted_grade_2019"

* Full control set (primary spec)
global controls "$ctrl_cont $ctrl_bin $ctrl_ofsted"

* Controls without Ofsted grade (sensitivity — keeps all 102 schools)
global controls_ngrade "$ctrl_cont $ctrl_bin"

* Primary outcome variables: overall P8 first, then 4 components
global outcomes "p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg"

* Culture predictors
global W  "gs_warmth_visit"
global S  "gs_strictness_visit"
global T  "gs_teaching_visit"
global WS "gs_warmth_visit gs_strictness_visit"

display "Macros defined."

Macros defined.


In [3]:
* ---- 3. Descriptive check on Tier 1 sample ----
preserve
keep if tier1

display _newline "=== Tier 1 descriptive statistics (N=102 visited schools) ==="
summarize $WS $T $outcomes ks2 fsm eal sen log_size academy urban_bin selective ///
    years_since_ofsted ofsted_grade_2019

display _newline "Pre-COVID Ofsted grade distribution:"
tabulate ofsted_grade_2019, missing

display _newline "Pairwise correlations (culture predictors and main outcomes):"
correlate $WS $T $outcomes

restore

(Note: Below code run with echo to enable preserve/restore functionality.)




(3,230 observations deleted)


=== Tier 1 descriptive statistics (N=102 visited schools) ===


    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
gs_warmth_~t |        102    6.615408    1.004875   4.541667   8.833333
gs_strictn~t |        102    7.091337    .8913628    4.88125        9.1
gs_teachin~t |        102    6.736366    .8460518   4.688889   8.833333
   p8mea_avg |        102     .259902    .4724913        -.7      1.345
p8meaeng_avg |        102    .2630392    .5050763       -.75      1.545
-------------+---------------------------------------------------------
p8meamat_avg |        102    .2765686    .4464586      -.595      1.345
p8meaebac_~g |        102    .3067647    .5428052       -.79       1.49
p8meaopen_~g |        102    .1817647    .5203482     -1.095      1.435
         ks2 |        102    104.9167    2.908509       99.8        117
         fsm |        102    29.16765  

In [4]:
* ================================================================
* PRIMARY REGRESSIONS — TIER 1 (n ≈ 95, dropped 7 with no pre-COVID grade)
* Stage 1: Total culture effect  y = α + β₁W + β₂S + X'γ + ε
* ================================================================
estimates clear

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store s1_`lbl'
    display _newline "Stage 1 — `lbl' (W+S, n=" e(N) "):"
    display "  β_W = " %6.3f _b[gs_warmth_visit] ///
            "  (se=" %6.3f _se[gs_warmth_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_warmth_visit]/_se[gs_warmth_visit])))
    display "  β_S = " %6.3f _b[gs_strictness_visit] ///
            "  (se=" %6.3f _se[gs_strictness_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_strictness_visit]/_se[gs_strictness_visit])))
    display "  R² = " %6.4f e(r2)
}


Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5952
                                                Root MSE          =     .32043



------------------------------------------------------------------------------


             |             Robust HC3


   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]


-------------+----------------------------------------------------------------


...


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =       6.89
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5616
                                                Root MSE          =     .35866

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1417135    .043923     3.23   0.002     .0543205    .2291064
gs_strictn~t |   .0792188   .0542554     1.46   0.148    -.0287325      .18717
         ks2 |   .0587733   .0246862     2.38   0.020     .0096555    .1078911
         fsm |  -.0056789   .0048108    -1.18   0.241    -.0152509     .003893
         eal


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =      10.03
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5057
                                                Root MSE          =      .3426

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0963812   .0579606     1.66   0.100    -.0189422    .2117045
gs_strictn~t |   .1068404   .0510776     2.09   0.040      .005212    .2084687
         ks2 |   .0394035   .0176788     2.23   0.029     .0042283    .0745788
         fsm |  -.0088499   .0039605    -2.23   0.028      -.01673   -.0009698
         eal


Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5704
                                                Root MSE          =     .38583

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1458399   .0599625     2.43   0.017     .0265334    .2651464
gs_strictn~t |   .1548066   .0578116     2.68   0.009     .0397796    .2698335
         ks2 |   .0592719   .0244213     2.43   0.017     .0106812    .1078625
         fsm |  -.0075709   .0046497    -1.63   0.107    -.0168224    .0016805
         eal


Stage 1 — EBaC (W+S, n=95):
 β_W =  0.146 (se= 0.060) p=0.017
 β_S =  0.155 (se= 0.058) p=0.009
 R² = 0.5704

Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5157
                                                Root MSE          =     .37151



------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1243832   .0522875     2.38   0.020     .0203476    .2284189
gs_strictn~t |   .1225779   .0598856     2.05   0.044     .0034244    .2417314
         ks2 |   .0651215   .0212488     3.06   0.003      .022843       .1074
         fsm |   -.005527     .00473    -1.17   0.246    -.0149383    .0038842
         eal |   .0068437   .0027033     2.53   0.013      .001465    .0122225
         sen |   .0043785   .0069276     0.63   0.529    -.0094051    .0181622
    log_size |   .0602892   .1183337     0.51   0.612    -.1751578    .2957362
years_sinc~d |  -.0104628   .0112831    -0.93   0.357    -.0329127    .0119871
     academy |  -.0189095   .1167352    -0.16   0.872    -.2511759     .213357
   urban_bin |

In [5]:
* ================================================================
* Stage 2: Teaching quality benchmark  y = α + γT + X'γ + ε
* ================================================================
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $T $controls if tier1, vce(hc3)
    estimates store s2_`lbl'
    display _newline "Stage 2 — `lbl' (T only, n=" e(N) "):"
    display "  γ_T = " %6.3f _b[gs_teaching_visit] ///
            "  (se=" %6.3f _se[gs_teaching_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_teaching_visit]/_se[gs_teaching_visit])))
    display "  R² = " %6.4f e(r2)
}


Linear regression                               Number of obs     =         95
                                                F(11, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5346
                                                Root MSE          =     .34148



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .2189295   .0557593     3.93   0.000     .1080066    .3298525
         ks2 |   .0450535   .0198455     2.27   0.026     .0055746    .0845325
         fsm |  -.0110672   .0043882    -2.52   0.014    -.0197966   -.0023378
         eal |   .0124061   .0020017     6.20   0.000     .0084242    .0163881
         sen |   .0078855   .0057521     1.37   0.174    -.0035573    .0193283
    log_size |  -.0705893   .1141906    -0.62   0.538    -.2977509    .1565722
years_sinc~d |  -.0048346   .0106012    -0.46   0.650    -.0259236    .0162545
     academy |  -.0149732   .1086862    -0.14   0.891    -.2311848    .2012383
   urban_bin |   .0672095   .1157089     0.58   0.563    -.1629724    .2973914
   selective |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .1909354   .0573494     3.33   0.001     .0768493    .3050216
         ks2 |   .0480655   .0253406     1.90   0.061     -.002345     .098476
         fsm |  -.0090508   .0055664    -1.63   0.108    -.0201242    .0020226
         eal |   .0144359   .0024685     5.85   0.000     .0095254    .0193465
         sen |   .0077597   .0061574     1.26   0.211    -.0044893    .0200087
    log_size |  -.1312791   .1278059    -1.03   0.307    -.3855256    .1229675
years_sinc~d |  -.0019514    .011313    -0.17   0.863    -.0244565    .0205537
     academy |   .0487235   .1197081     0.41   0.685     -.189414    .2868611
   urban_bin |   .0863366   .1373657     0.63   0.531    -.1869275    .3596006
   selective |


Linear regression                               Number of obs     =         95
                                                F(12, 82)         =       7.67
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4621
                                                Root MSE          =     .35518

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .1796972    .062186     2.89   0.005     .0559896    .3034049
         ks2 |   .0296217    .017385     1.70   0.092    -.0049626    .0642059
         fsm |  -.0124482   .0039569    -3.15   0.002    -.0203198   -.0045767
         eal |   .0125086   .0018301     6.83   0.000      .008868    .0161492
         sen


Stage 2 — Maths (T only, n=95):
 γ_T =  0.180 (se= 0.062) p=0.005
 R² = 0.4621

Linear regression                               Number of obs     =         95
                                                F(12, 82)         =       6.88
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4871
                                                Root MSE          =     .41897



------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .2469758   .0697557     3.54   0.001     .1082094    .3857422
         ks2 |   .0469523   .0241423     1.94   0.055    -.0010744     .094979
         fsm |  -.0123085   .0054004    -2.28   0.025    -.0230516   -.0015655
         eal |    .014508   .0025294     5.74   0.000     .0094762    .0195398
         sen |   .0090601   .0075788     1.20   0.235    -.0060165    .0241367
    log_size |  -.1089144   .1566859    -0.70   0.489    -.4206127    .2027838
years_sinc~d |  -.0035722   .0128653    -0.28   0.782    -.0291653     .022021
     academy |  -.0091461   .1315453    -0.07   0.945    -.2708316    .2525394
   urban_bin |   .0373658   .1520753     0.25   0.807    -.2651604     .339892
   selective |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .2414549   .0603431     4.00   0.000     .1214134    .3614965
         ks2 |   .0504227   .0213229     2.36   0.020     .0080047    .0928407
         fsm |  -.0104912   .0049598    -2.12   0.037     -.020358   -.0006245
         eal |   .0092019    .002839     3.24   0.002     .0035542    .0148497
         sen |   .0044144    .006456     0.68   0.496    -.0084286    .0172575
    log_size |  -.0028611   .1207271    -0.02   0.981    -.2430257    .2373036
years_sinc~d |  -.0068117   .0113156    -0.60   0.549    -.0293221    .0156987
     academy |  -.0516415   .1193306    -0.43   0.666    -.2890281    .1857452
   urban_bin |   .0843407   .1239383     0.68   0.498     -.162212    .3308934
   selective |

In [6]:
* ================================================================
* Stage 3: Direct culture effect net of teaching  y = α + β₁W + β₂S + γT + X'γ + ε
* ================================================================
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS $T $controls if tier1, vce(hc3)
    estimates store s3_`lbl'
    display _newline "Stage 3 — `lbl' (W+S+T, n=" e(N) "):"
    display "  β_W = " %6.3f _b[gs_warmth_visit] ///
            "  (se=" %6.3f _se[gs_warmth_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_warmth_visit]/_se[gs_warmth_visit])))
    display "  β_S = " %6.3f _b[gs_strictness_visit] ///
            "  (se=" %6.3f _se[gs_strictness_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_strictness_visit]/_se[gs_strictness_visit])))
    display "  γ_T = " %6.3f _b[gs_teaching_visit] ///
            "  (se=" %6.3f _se[gs_teaching_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_teaching_visit]/_se[gs_teaching_visit])))
    display "  R² = " %6.4f e(r2)
}


Linear regression                               Number of obs     =         95
                                                F(14, 80)         =      10.36
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5955
                                                Root MSE          =     .32231



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1185877   .0562942     2.11   0.038     .0065588    .2306167
gs_strictn~t |   .1156202   .0469464     2.46   0.016     .0221938    .2090466
gs_teachin~t |   .0181015   .0788936     0.23   0.819    -.1389018    .1751047
         ks2 |   .0557147   .0209147     2.66   0.009     .0140931    .0973363
         fsm |  -.0071864    .004278    -1.68   0.097    -.0156999     .001327
         eal |   .0106122   .0018262     5.81   0.000      .006978    .0142464
         sen |   .0081975   .0057041     1.44   0.155    -.0031541     .019549
    log_size |  -.0213021   .1099858    -0.19   0.847    -.2401809    .1975767
years_sinc~d |  -.0078899   .0103588    -0.76   0.449    -.0285045    .0127248
     academy |


Linear regression                               Number of obs     =         95
                                                F(14, 80)         =       6.35
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5617
                                                Root MSE          =     .36086

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1470918    .065108     2.26   0.027     .0175228    .2766609
gs_strictn~t |   .0818777   .0518142     1.58   0.118    -.0212358    .1849912
gs_teachin~t |  -.0111035   .0910658    -0.12   0.903    -.1923302    .1701232
         ks2 |    .059652   .0264128     2.26   0.027     .0070887    .1122152
         fsm

       _cons |  -7.116225    3.03808    -2.34   0.022     -13.1622   -1.070253
------------------------------------------------------------------------------

Stage 3 — English (W+S+T, n=95):
 β_W =  0.147 (se= 0.065) p=0.027
 β_S =  0.082 (se= 0.052) p=0.118
 γ_T = -0.011 (se= 0.091) p=0.903
 R² = 0.5617

Linear regression                               Number of obs     =         95
                                                F(14, 80)         =       9.26
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5060
                                                Root MSE          =     .34462



------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0874011   .0641853     1.36   0.177    -.0403317    .2151339
gs_strictn~t |   .1024009   .0575605     1.78   0.079    -.0121482    .2169499
gs_teachin~t |   .0185391   .0890012     0.21   0.836    -.1585789    .1956571
         ks2 |   .0379365      .0189     2.01   0.048     .0003243    .0755486
         fsm |  -.0092527   .0041641    -2.22   0.029    -.0175395    -.000966
         eal |   .0109681   .0019461     5.64   0.000     .0070952     .014841
         sen |   .0105338   .0057358     1.84   0.070    -.0008807    .0219484
    log_size |  -.0132809   .1237604    -0.11   0.915    -.2595719    .2330101
years_sinc~d |  -.0090565   .0113996    -0.79   0.429    -.0317424    .0136295
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1589097   .0751647     2.11   0.038     .0093273    .3084922
gs_strictn~t |   .1612679   .0583337     2.76   0.007     .0451802    .2773557
gs_teachin~t |  -.0269823   .0970049    -0.28   0.782    -.2200283    .1660636
         ks2 |    .061407   .0260353     2.36   0.021     .0095952    .1132189
         fsm |  -.0069846   .0050203    -1.39   0.168    -.0169753     .003006
         eal |   .0120237   .0021239     5.66   0.000     .0077969    .0162504
         sen |   .0094383   .0073364     1.29   0.202    -.0051615    .0240381
    log_size |  -.0404257   .1496268    -0.27   0.788    -.3381926    .2573411
years_sinc~d |  -.0076814   .0122634    -0.63   0.533    -.0320864    .0167236
     academy |


Stage 3 — EBaC (W+S+T, n=95):
 β_W =  0.159 (se= 0.075) p=0.038
 β_S =  0.161 (se= 0.058) p=0.007
 γ_T = -0.027 (se= 0.097) p=0.782
 R² = 0.5708

Linear regression                               Number of obs     =         95
                                                F(14, 80)         =       7.47
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5213
                                                Root MSE          =     .37167

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0840165   .0632428     1.33   0.188    -.0418406    .2098737
gs_strictn~t |   .1026217   .0633223     1.62   0.109    -.0233937    .2286372
gs_teachin~t |   .083335

         eal |   .0076679   .0030474     2.52   0.014     .0016034    .0137324
         sen |   .0045052   .0068136     0.66   0.510    -.0090543    .0180648
    log_size |   .0400432   .1243149     0.32   0.748    -.2073512    .2874377
years_sinc~d |  -.0090256   .0116242    -0.78   0.440    -.0321585    .0141072
     academy |  -.0270683   .1167076    -0.23   0.817     -.259324    .2051873
   urban_bin |   .0855872    .119586     0.72   0.476    -.1523965    .3235709
   selective |   .1753935   .4215007     0.42   0.678    -.6634196    1.014207
             |
ofsted_~2019 |
          3  |  -.1752234   .2038574    -0.86   0.393    -.5809126    .2304658
          4  |  -.1690574   .3041233    -0.56   0.580     -.774282    .4361672
             |
       _cons |  -8.271729   2.662629    -3.11   0.003    -13.57053   -2.972928
------------------------------------------------------------------------------

Stage 3 — Open (W+S+T, n=95):
 β_W =  0.084 (se= 0.063) p=0.188
 β_S =  0.103 (se= 0.

In [7]:
* ================================================================
* Attenuation summary: Stage 1 → Stage 3 coefficient retention
* ================================================================
display _newline "=== Attenuation: Stage 1 β retained in Stage 3 ==="
display " Outcome     β_W(S1)  β_W(S3)  %ret   β_S(S1)  β_S(S3)  %ret"
display " -----------------------------------------------------------------"

foreach lbl in Overall English Maths EBaC Open {
    estimates restore s1_`lbl'
    local bw1 = _b[gs_warmth_visit]
    local bs1 = _b[gs_strictness_visit]
    
    estimates restore s3_`lbl'
    local bw3 = _b[gs_warmth_visit]
    local bs3 = _b[gs_strictness_visit]
    
    local retW = cond(`bw1'!=0, `bw3'/`bw1'*100, .)
    local retS = cond(`bs1'!=0, `bs3'/`bs1'*100, .)
    
    display " `lbl': " ///
        %7.3f `bw1' "  " %7.3f `bw3' "  " %5.0f `retW' "%%" ///
        "   " %7.3f `bs1' "  " %7.3f `bs3' "  " %5.0f `retS' "%"
}


=== Attenuation: Stage 1 β retained in Stage 3 ===
 Outcome β_W(S1) β_W(S3) %ret β_S(S1) β_S(S3) %ret
 -----------------------------------------------------------------
(results s1_Overall are active now)
(results s3_Overall are active now)
 Overall:   0.127   0.119    93%%   0.120   0.116    96%
(results s1_English are active now)
(results s3_English are active now)
 English:   0.142   0.147   104%%   0.079   0.082   103%


(results s1_Maths are active now)
(results s3_Maths are active now)
 Maths:   0.096   0.087    91%%   0.107   0.102    96%
(results s1_EBaC are active now)
(results s3_EBaC are active now)
 EBaC:   0.146   0.159   109%%   0.155   0.161   104%
(results s1_Open are active now)
(results s3_Open are active now)
 Open:   0.124   0.084    68%%   0.123   0.103    84%


In [8]:
* ---- VIF check (last Stage 3 regression) ----
regress p8meaebac_avg $WS $T $controls if tier1, vce(hc3)
estat vif


Linear regression                               Number of obs     =         95
                                                F(14, 80)         =       8.53
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5708
                                                Root MSE          =     .38802



------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1589097   .0751647     2.11   0.038     .0093273    .3084922
gs_strictn~t |   .1612679   .0583337     2.76   0.007     .0451802    .2773557
gs_teachin~t |  -.0269823   .0970049    -0.28   0.782    -.2200283    .1660636
         ks2 |    .061407   .0260353     2.36   0.021     .0095952    .1132189
         fsm |  -.0069846   .0050203    -1.39   0.168    -.0169753     .003006
         eal |   .0120237   .0021239     5.66   0.000     .0077969    .0162504
         sen |   .0094383   .0073364     1.29   0.202    -.0051615    .0240381
    log_size |  -.0404257   .1496268    -0.27   0.788    -.3381926    .2573411
years_sinc~d |  -.0076814   .0122634    -0.63   0.533    -.0320864    .0167236
     academy |


    Variable |       VIF       1/VIF  
-------------+----------------------


gs_warmth_~t |      3.07    0.326096
gs_strictn~t |      1.86    0.538133
gs_teachin~t |      3.82    0.261868
         ks2 |      2.15    0.464163
         fsm |      3.26    0.306625
         eal |      2.53    0.395404
         sen |      1.56    0.642121
    log_size |      1.29    0.775050
years_sinc~d |      1.29    0.775142
     academy |      1.24    0.809144
   urban_bin |      1.29    0.777383
   selective |      1.21    0.828959
ofsted_~2019 |
          3  |      1.34    0.744012
          4  |      1.10    0.909052
-------------+----------------------
    Mean VIF |      1.93


In [9]:
* ================================================================
* Export: tab_main_results.tex
* Three-panel table: Stage 1 (5 cols) | Stage 2 (5 cols) | Stage 3 (5 cols)
* ================================================================

* Panel A: Stage 1
esttab s1_Overall s1_English s1_Maths s1_EBaC s1_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_main_results_s1.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "Warmth (\$W\$)" gs_strictness_visit "Strictness (\$S\$)") ///
    stats(N r2, labels("\$N\$" "\$R^2\$") fmt(%9.0f %8.3f)) ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    title("Stage 1: Total culture effect (\$W + S\$)") ///
    b(%8.3f) se(%8.3f)

* Panel B: Stage 2
esttab s2_Overall s2_English s2_Maths s2_EBaC s2_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_main_results_s2.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_teaching_visit) ///
    coeflabels(gs_teaching_visit "Teaching quality (\$T\$)") ///
    stats(N r2, labels("\$N\$" "\$R^2\$") fmt(%9.0f %8.3f)) ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    title("Stage 2: Teaching quality benchmark (\$T\$)") ///
    b(%8.3f) se(%8.3f)

* Panel C: Stage 3
esttab s3_Overall s3_English s3_Maths s3_EBaC s3_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_main_results_s3.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit gs_teaching_visit) ///
    coeflabels(gs_warmth_visit "Warmth (\$W\$)" ///
               gs_strictness_visit "Strictness (\$S\$)" ///
               gs_teaching_visit "Teaching quality (\$T\$)") ///
    stats(N r2, labels("\$N\$" "\$R^2\$") fmt(%9.0f %8.3f)) ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    title("Stage 3: Culture net of teaching (\$W + S + T\$)") ///
    b(%8.3f) se(%8.3f)

display "Main results tables written."

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_main_results_s1.tex)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_main_results_s2.tex)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_main_results_s3.tex)
Main results tables written.


In [10]:
* ================================================================
* ROBUSTNESS 1: No pre-COVID Ofsted grade (all 102 schools)
* ================================================================
estimates clear

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls_ngrade if tier1, vce(hc3)
    estimates store rob_ngrade_`lbl'
    display "No-grade Stage 1 — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

display _newline "(Compare to primary spec with Ofsted grade controls above)"


Linear regression                               Number of obs     =        101
                                                F(11, 89)         =      12.04
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5679
                                                Root MSE          =     .32938



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1418725   .0449299     3.16   0.002     .0525978    .2311473
gs_strictn~t |   .0986204   .0444003     2.22   0.029     .0103979    .1868428
         ks2 |    .059291   .0182979     3.24   0.002     .0229335    .0956484
         fsm |  -.0076548   .0037541    -2.04   0.044     -.015114   -.0001956
         eal |   .0103816   .0016146     6.43   0.000     .0071733    .0135899
         sen |   .0076494   .0055667     1.37   0.173    -.0034114    .0187103
    log_size |   .0284535   .0997155     0.29   0.776    -.1696791    .2265861
years_sinc~d |  -.0025979   .0090226    -0.29   0.774    -.0205256    .0153298
     academy |  -.0336198   .1007035    -0.33   0.739    -.2337155    .1664759
   urban_bin |

   .0584682   .0506843     1.15   0.252    -.0422405    .1591769
         ks2 |   .0605799   .0225007     2.69   0.008     .0158714    .1052883
         fsm |  -.0063532   .0045615    -1.39   0.167    -.0154167    .0027104
         eal |   .0130018   .0020289     6.41   0.000     .0089704    .0170333
         sen |   .0082906    .005899     1.41   0.163    -.0034307    .0200118
    log_size |  -.0461131   .1078863    -0.43   0.670    -.2604809    .1682547
years_sinc~d |  -.0023064   .0099343    -0.23   0.817    -.0220455    .0174328
     academy |   .0187432   .1121002     0.17   0.868    -.2039976    .2414839
   urban_bin |   .1821486   .1316619     1.38   0.170    -.0794608    .4437581
   selective |   .2217643   .2713254     0.82   0.416    -.3173536    .7608822
       _cons |  -7.870866    2.64194    -2.98   0.004    -13.12034   -2.621388
------------------------------------------------------------------------------
No-grade Stage 1 — English (n=101): β_W= 0.152 β_S= 0.058

Linear 

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1050091   .0539249     1.95   0.055    -.0021385    .2121567
gs_strictn~t |   .0910766   .0480259     1.90   0.061    -.0043498    .1865031
         ks2 |   .0461946   .0168512     2.74   0.007     .0127116    .0796775
         fsm |   -.009426   .0037787    -2.49   0.014    -.0169342   -.0019178
         eal |   .0103876   .0016125     6.44   0.000     .0071835    .0135917
         sen |   .0104512   .0056488     1.85   0.068    -.0007728    .0216752
    log_size |   .0051096    .110302     0.05   0.963    -.2140581    .2242774
years_sinc~d |  -.0027426   .0098328    -0.28   0.781    -.0222803     .016795
     academy |  -.0619545   .1075531    -0.58   0.566    -.2756603    .1517513
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1653609   .0567937     2.91   0.005     .0525131    .2782087
gs_strictn~t |   .1267978   .0545932     2.32   0.022     .0183222    .2352734
         ks2 |    .063491   .0221276     2.87   0.005      .019524     .107458
         fsm |  -.0082236   .0044921    -1.83   0.070    -.0171493    .0007021
         eal |   .0122423   .0020186     6.06   0.000     .0082313    .0162532
         sen |   .0091863   .0068857     1.33   0.186    -.0044955    .0228681
    log_size |  -.0041239   .1358631    -0.03   0.976     -.274081    .2658331
years_sinc~d |  -.0004535   .0107265    -0.04   0.966    -.0217669    .0208599
     academy |  -.0179226   .1165361    -0.15   0.878    -.2494774    .2136322
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .140642   .0539234     2.61   0.011     .0334974    .2477866
gs_strictn~t |    .103152   .0561593     1.84   0.070    -.0084354    .2147394
         ks2 |   .0630885   .0195847     3.22   0.002     .0241741     .102003
         fsm |  -.0069315   .0042903    -1.62   0.110    -.0154564    .0015933
         eal |   .0070212    .002369     2.96   0.004      .002314    .0117284
         sen |   .0032241   .0068294     0.47   0.638    -.0103457     .016794
    log_size |   .1234086   .1088599     1.13   0.260    -.0928938     .339711
years_sinc~d |   -.004699   .0101065    -0.46   0.643    -.0247804    .0153824
     academy |   -.069834   .1175181    -0.59   0.554      -.30334    .1636719
   urban_bin |

In [11]:
* ================================================================
* ROBUSTNESS 2: Single year 2023-24 P8 (overall + components)
* ================================================================

foreach outcome in p8mea_2324 p8meaeng_2324 p8meamat_2324 p8meaebac_2324 p8meaopen_2324 {
    local lbl = cond("`outcome'"=="p8mea_2324",    "Overall", ///
                cond("`outcome'"=="p8meaeng_2324",  "English", ///
                cond("`outcome'"=="p8meamat_2324",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_2324", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store rob_2324_`lbl'
    display "2023-24 single year — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =      11.23
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6009
                                                Root MSE          =     .32866



------------------------------------------------------------------------------
             |             Robust HC3
  p8mea_2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1375959   .0489673     2.81   0.006     .0401663    .2350255
gs_strictn~t |    .118503    .047217     2.51   0.014     .0245561      .21245
         ks2 |   .0566726   .0210245     2.70   0.009     .0148404    .0985047
         fsm |  -.0081703   .0042921    -1.90   0.061    -.0167102    .0003695
         eal |   .0105051   .0017504     6.00   0.000     .0070223     .013988
         sen |   .0065191   .0057314     1.14   0.259    -.0048845    .0179227
    log_size |  -.0611239   .0993681    -0.62   0.540    -.2588353    .1365875
years_sinc~d |  -.0054473   .0106818    -0.51   0.611    -.0267007    .0158061
     academy |    .009172   .1010696     0.09   0.928    -.1919247    .2102687
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaen~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .140881   .0491042     2.87   0.005     .0431791    .2385829
gs_strictn~t |    .076271   .0560059     1.36   0.177    -.0351633    .1877052
         ks2 |   .0574783   .0246534     2.33   0.022     .0084258    .1065308
         fsm |  -.0056689   .0050459    -1.12   0.265    -.0157087    .0043709
         eal |   .0125848   .0021955     5.73   0.000     .0082164    .0169533
         sen |   .0048012   .0062415     0.77   0.444    -.0076175    .0172199
    log_size |  -.1206725   .1186566    -1.02   0.312     -.356762     .115417
years_sinc~d |  -.0026729   .0117678    -0.23   0.821    -.0260871    .0207414
     academy |   .0409071   .1124481     0.36   0.717    -.1828293    .2646436
   urban_bin |


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =       8.85
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4895
                                                Root MSE          =     .35713

------------------------------------------------------------------------------
             |             Robust HC3
p8meama~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |

    .103063   .0622066     1.66   0.101    -.0207087    .2268347
gs_strictn~t |   .0957278   .0514005     1.86   0.066     -.006543    .1979986
         ks2 |   .0400099   .0195112     2.05   0.044     .0011888     .078831
         fsm |  -.0102404   .0045015    -2.27   0.026    -.0191971   -.0012838
         eal |   .0106539   .0019519     5.46   0.000     .0067701    .0145377
         sen |   .0101687   .0061731     1.65   0.103    -.0021138    .0224513
    log_size |   -.042753   .1093692    -0.39   0.697    -.2603635    .1748575
years_sinc~d |  -.0059084   .0117233    -0.50   0.616    -.0292341    .0174172
     academy |  -.0159176   .1115316    -0.14   0.887    -.2378305    .2059954
   urban_bin |    .070065   .1088902     0.64   0.522    -.1465923    .2867224
   selective |  -.2296517   .2262982    -1.01   0.313    -.6799142    .2206109
             |
ofsted_~2019 |
          3  |  -.2093025   .1182298    -1.77   0.080    -.4445427    .0259377
          4  |    -.25289   .1570703

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeb~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1508525   .0646811     2.33   0.022     .0221575    .2795475
gs_strictn~t |   .1660517   .0594777     2.79   0.007     .0477097    .2843937
         ks2 |   .0607865    .026611     2.28   0.025      .007839    .1137339
         fsm |  -.0080153   .0050003    -1.60   0.113    -.0179644    .0019338
         eal |   .0118717   .0021952     5.41   0.000     .0075039    .0162395
         sen |   .0074553   .0073845     1.01   0.316    -.0072376    .0221482
    log_size |  -.0733628   .1356054    -0.54   0.590     -.343175    .1964493
years_sinc~d |  -.0050805   .0126626    -0.40   0.689    -.0302752    .0201142
     academy |    .008911   .1175353     0.08   0.940    -.2249474    .2427693
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaop~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1480199   .0558273     2.65   0.010     .0369412    .2590986
gs_strictn~t |   .1141452   .0615281     1.86   0.067    -.0082763    .2365668
         ks2 |    .060537   .0219114     2.76   0.007     .0169401    .1041338
         fsm |  -.0090478   .0050583    -1.79   0.077    -.0191123    .0010167
         eal |   .0080324   .0027034     2.97   0.004     .0026535    .0134113
         sen |   .0039023   .0069715     0.56   0.577    -.0099689    .0177734
    log_size |  -.0252376   .1162051    -0.22   0.829    -.2564493    .2059742
years_sinc~d |  -.0085178   .0119511    -0.71   0.478    -.0322967     .015261
     academy |   -.006115   .1237168    -0.05   0.961    -.2522726    .2400427
   urban_bin |

In [12]:
* ================================================================
* ROBUSTNESS 3: Att8 2024-25 components + total (contemporaneous)
* att8_total = sum of 4 bucket scores (already destringed in cell-load)
* ================================================================

gen att8_total_2425 = att8screng_2425 + att8scrmat_2425 + att8screbac_2425 + att8scropen_2425

foreach outcome in att8_total_2425 att8screng_2425 att8scrmat_2425 att8screbac_2425 att8scropen_2425 {
    local lbl = cond("`outcome'"=="att8_total_2425",  "Overall", ///
                cond("`outcome'"=="att8screng_2425",  "English", ///
                cond("`outcome'"=="att8scrmat_2425",  "Maths",   ///
                cond("`outcome'"=="att8screbac_2425", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store rob_att8_`lbl'
    display "Att8 2024-25 — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

display "(Note: Att8 does not subtract KS2 prior attainment — higher R² expected)"

(58 missing values generated)

Linear regression                               Number of obs     =         95
                                                F(13, 81)         =      52.82
                                                Prob > F          =     0.0000
                                                R-squared         =     0.8825
                                                Root MSE          =     3.6202



------------------------------------------------------------------------------
             |             Robust HC3
att8_to~2425 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .8947873   .5016396     1.78   0.078    -.1033181    1.892893
gs_strictn~t |   1.766748   .4764711     3.71   0.000     .8187198    2.714776
         ks2 |   2.791455   .1956929    14.26   0.000     2.402087    3.180822
         fsm |  -.0969361   .0523944    -1.85   0.068    -.2011846    .0073124
         eal |   .1040618   .0234202     4.44   0.000      .057463    .1506606
         sen |   .1310702   .0605052     2.17   0.033     .0106838    .2514567
    log_size |  -.1182355    1.57677    -0.07   0.940    -3.255513    3.019042
years_sinc~d |  -.0818553   .1095593    -0.75   0.457     -.299844    .1361334
     academy |  -1.032784    1.15394    -0.90   0.373    -3.328762    1.263194
   urban_bin |


Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.8574
                                                Root MSE          =     .81433

------------------------------------------------------------------------------
             |             Robust HC3
att8scrmat~5 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0432291   .1365313     0.32   0.752    -.2284253    .3148835
gs_strictn~t |   .4254159   .1237554     3.44   0.001     .1791815    .6716504
         ks2 |   .5713844   .0415107    13.76   0.000      .488791    .6539778
         fsm |  -.0175127   .0105503    -1.66   0.101    -.0385045    .0034791
         eal

   urban_bin |  -.1089049   .2922545    -0.37   0.710    -.6903997    .4725898
   selective |  -.3174735   .6671133    -0.48   0.635     -1.64482    1.009873
             |
ofsted_~2019 |
          3  |  -.1408606   .2379159    -0.59   0.555    -.6142385    .3325174
          4  |  -.5553204   .3103759    -1.79   0.077    -1.172871    .0622304
             |
       _cons |  -54.14526    5.24774   -10.32   0.000    -64.58661    -43.7039
------------------------------------------------------------------------------
Att8 2024-25 — Maths (n=95): β_W= 0.043 β_S= 0.425

Linear regression                               Number of obs     =         95
                                                F(13, 81)         =      43.01
                                                Prob > F          =     0.0000
                                                R-squared         =     0.8643
                                                Root MSE          =       1.29



------------------------------------------------------------------------------
             |             Robust HC3
att8screba~5 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .2476399   .1716293     1.44   0.153    -.0938487    .5891284
gs_strictn~t |   .6773388   .1737799     3.90   0.000     .3315714    1.023106
         ks2 |   .9088253   .0679398    13.38   0.000     .7736464    1.044004
         fsm |  -.0315738   .0179255    -1.76   0.082      -.06724    .0040924
         eal |   .0385407   .0081493     4.73   0.000     .0223261    .0547554
         sen |   .0421079   .0230106     1.83   0.071    -.0036761    .0878918
    log_size |  -.0849015   .5966209    -0.14   0.887     -1.27199    1.102187
years_sinc~d |  -.0232635   .0373791    -0.62   0.535    -.0976363    .0511092
     academy |  -.3802837   .3926705    -0.97   0.336    -1.161575    .4010075
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
att8scrope~5 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .3604168   .1636591     2.20   0.030     .0347864    .6860471
gs_strictn~t |    .426775    .150053     2.84   0.006     .1282167    .7253334
         ks2 |   .8142924   .0617725    13.18   0.000     .6913845    .9372003
         fsm |  -.0315852   .0172407    -1.83   0.071    -.0658888    .0027184
         eal |    .025108   .0087579     2.87   0.005     .0076825    .0425334
         sen |   .0351159   .0178639     1.97   0.053    -.0004276    .0706594
    log_size |    .146782   .4624569     0.32   0.752    -.7733622    1.066926
years_sinc~d |   -.045728   .0322634    -1.42   0.160    -.1099221    .0184661
     academy |  -.4273817    .363597    -1.18   0.243    -1.150826    .2960622
   urban_bin |

In [13]:
* ================================================================
* ROBUSTNESS 4: Continuity-restricted sample
* (ofsted_HeadteacherChanged == 0: HT unchanged since Ofsted inspection)
* Note: only 30 of 102 visited schools have this variable; not missing at random
* ================================================================

* ofsted_headteacherchanged is stored as "True"/"False" strings in the CSV
gen ht_changed = .
replace ht_changed = 1 if lower(ofsted_headteacherchanged) == "true"
replace ht_changed = 0 if lower(ofsted_headteacherchanged) == "false"

count if tier1 & ht_changed == 0
count if tier1 & ht_changed == 1
count if tier1 & missing(ht_changed)

foreach outcome in p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8meaeng_avg", "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open")))
    
    * Note: using controls_ngrade to avoid further sample reduction from missing grade
    regress `outcome' $WS $controls_ngrade if tier1 & ht_changed==0, vce(hc3)
    estimates store rob_cont_`lbl'
    display "Continuity (unchanged HT) — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

display _newline "Caution: n is very small in this restricted sample — interpret as directional only."

(3,332 missing values generated)
(636 real changes made)
(786 real changes made)
  23
  7
  72

Linear regression                               Number of obs     =         23
                                                F(11, 11)         =       1.17
                                                Prob > F          =     0.3980
                                                R-squared         =     0.6931
                                                Root MSE          =     .39257



------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    -.04871   .2063203    -0.24   0.818     -.502818    .4053979
gs_strictn~t |   .3327074   .2784601     1.19   0.257    -.2801792     .945594
         ks2 |   .1045867   .1311403     0.80   0.442    -.1840512    .3932245
         fsm |  -.0062608   .0189284    -0.33   0.747     -.047922    .0354004
         eal |   .0191271   .0082976     2.31   0.042     .0008642      .03739
         sen |   .0300736   .0461152     0.65   0.528    -.0714253    .1315726
    log_size |  -.0574057   .8134815    -0.07   0.945    -1.847866    1.733055
years_sinc~d |  -.0064898   .0400615    -0.16   0.874    -.0946645     .081685
     academy |   .1542103   .5272794     0.29   0.775    -1.006324    1.314744
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0546979   .1680987    -0.33   0.751    -.4246806    .3152849
gs_strictn~t |   .2072467   .2198563     0.94   0.366    -.2766537     .691147
         ks2 |  -.0380629   .0977097    -0.39   0.704    -.2531206    .1769947
         fsm |  -.0134195     .01695    -0.79   0.445    -.0507262    .0238872
         eal |   .0063459   .0068669     0.92   0.375     -.008768    .0214598
         sen |   .0072161   .0378796     0.19   0.852    -.0761564    .0905885
    log_size |  -.1503319   .6419338    -0.23   0.819    -1.563219    1.262555
years_sinc~d |   .0139906   .0323304     0.43   0.674    -.0571681    .0851494
     academy |   .2782096    .516516     0.54   0.601    -.8586345    1.415054
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0483485   .2338107    -0.21   0.840    -.5629624    .4662654
gs_strictn~t |    .242196   .3241881     0.75   0.471    -.4713373    .9557293
         ks2 |   .0768108   .1526645     0.50   0.625    -.2592015    .4128231
         fsm |  -.0063531   .0192433    -0.33   0.747    -.0487074    .0360011
         eal |   .0111665   .0115284     0.97   0.354    -.0142073    .0365403
         sen |   .0208881   .0424472     0.49   0.632    -.0725376    .1143138
    log_size |  -.1899785   1.001828    -0.19   0.853    -2.394988    2.015031
years_sinc~d |   .0000444   .0474141     0.00   0.999    -.1043133    .1044022
     academy |   .2199353   .6760508     0.33   0.751    -1.268042    1.707913
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0495255   .2520901     0.20   0.848    -.5053211    .6043721
gs_strictn~t |   .2856897   .2625869     1.09   0.300    -.2922602    .8636395
         ks2 |   .0414306   .1187619     0.35   0.734    -.2199626    .3028237
         fsm |   -.013205   .0162725    -0.81   0.434    -.0490204    .0226105
         eal |   .0178234   .0062481     2.85   0.016     .0040714    .0315754
         sen |  -.0050868   .0444961    -0.11   0.911     -.103022    .0928485
    log_size |  -.7112985    .823759    -0.86   0.406     -2.52438    1.101783
years_sinc~d |  -.0084775   .0464649    -0.18   0.859    -.1107461    .0937911
     academy |   .0272256   .5544047     0.05   0.962    -1.193011    1.247462
   urban_bin |

In [14]:
* ================================================================
* ROBUSTNESS 5: W × S interaction term
* ================================================================

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' c.gs_warmth_visit##c.gs_strictness_visit $controls if tier1, vce(hc3)
    estimates store rob_inter_`lbl'
    display "W×S interaction — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit] ///
        " β_WxS=" %6.3f _b[c.gs_warmth_visit#c.gs_strictness_visit] ///
        " p_inter=" %5.3f (2*ttail(e(df_r), abs(_b[c.gs_warmth_visit#c.gs_strictness_visit]/_se[c.gs_warmth_visit#c.gs_strictness_visit])))
}


Linear regression                               Number of obs     =         95
                                                F(13, 80)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5994
                                                Root MSE          =     .32073



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.1061683   .2936309    -0.36   0.719    -.6905124    .4781758
gs_strictn~t |  -.0989537     .27016    -0.37   0.715    -.6365891    .4386818
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0333996   .0423438     0.79   0.433    -.0508672    .1176664
             |
         ks2 |   .0555978   .0202374     2.75   0.007     .0153242    .0958714
         fsm |  -.0068154   .0039568    -1.72   0.089    -.0146898     .001059
         eal |   .0101641   .0018831     5.40   0.000     .0064167    .0139116
         sen |   .0076441    .005917     1.29   0.200    -.0041311    .0194193
    log_size |  -.0039128   .1110378    -0.04   0.972    -.2248851    .2170595
years_sinc~d |  -.


Linear regression                               Number of obs     =         95
                                                F(14, 80)         =       8.79
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5630
                                                Root MSE          =     .36031

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0039349   .3071353    -0.01   0.990    -.6151537    .6072839
gs_strictn~t |  -.0573139   .2797077    -0.20   0.838    -.6139499    .4993221
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0208312   .0433427     0.48   0.632    -.0654234    .1070859
             |
         ks2 |  

W×S interaction — English (n=95): β_W=-0.004 β_S=-0.057 β_WxS= 0.021 p_inter=0.
> 632

Linear regression                               Number of obs     =         95
                                                F(14, 80)         =       8.30
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5121
                                                Root MSE          =     .34248

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------


gs_warmth_~t |  -.1817852    .339871    -0.53   0.594    -.8581501    .4945796
gs_strictn~t |  -.1539164   .3144127    -0.49   0.626    -.7796176    .4717847
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0397845   .0487017     0.82   0.416     -.057135     .136704
             |
         ks2 |    .037558   .0181505     2.07   0.042     .0014372    .0736787
         fsm |  -.0088765   .0040231    -2.21   0.030    -.0168827   -.0008702
         eal |   .0104642   .0018907     5.53   0.000     .0067016    .0142268
         sen |   .0098793   .0058999     1.67   0.098    -.0018619    .0216204
    log_size |   .0066983   .1192165     0.06   0.955    -.2305501    .2439468
years_sinc~d |  -.0088286    .011022    -0.80   0.426    -.0307631    .0131059
     academy |   .0066554    .115905     0.06   0.954    -.2240028    .2373137
   urban_bin |   .0781633   .1042375     0.75   0.456     -.129276    .2856026
   selective |   .0187609   .3157817     0.06   0.953   

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.2215925   .3409241    -0.65   0.518    -.9000531     .456868
gs_strictn~t |  -.1896294   .2970976    -0.64   0.525    -.7808726    .4016137
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0525517   .0476639     1.10   0.274    -.0423024    .1474059
             |
         ks2 |   .0568341    .024629     2.31   0.024     .0078208    .1058473
         fsm |   -.007606   .0045707    -1.66   0.100     -.016702      .00149
         eal |   .0118672    .002167     5.48   0.000     .0075547    .0161796
         sen |   .0086519    .007459     1.16   0.250    -.0061919    .0234958
    log_size |  -.0265395   .1474059    -0.18   0.858    -.3198867    .2668076
years_sinc~d |  -.

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0116357   .3423737     0.03   0.973    -.6697097    .6929811
gs_strictn~t |    .016887   .3291023     0.05   0.959    -.6380475    .6718214
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0161256   .0505115     0.32   0.750    -.0843955    .1166467
             |
         ks2 |   .0643735    .021482     3.00   0.004     .0216229     .107124
         fsm |  -.0055378   .0047866    -1.16   0.251    -.0150635    .0039879
         eal |   .0067138   .0029182     2.30   0.024     .0009064    .0125212
         sen |   .0041246    .007133     0.58   0.565    -.0100704    .0183197
    log_size |   .0665617   .1284883     0.52   0.606    -.1891382    .3222616
years_sinc~d |  -.

In [15]:
* ================================================================
* ROBUSTNESS 6: Add SEMH baseline 2016 as additional control
* ================================================================
destring semh_baseline_2016, replace force

count if tier1 & !missing(semh_baseline_2016)

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS semh_baseline_2016 $controls if tier1, vce(hc3)
    estimates store rob_semh_`lbl'
    display "SEMH control — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

semh_baseline_2016 already numeric; no replace
  95

Linear regression                               Number of obs     =         90
                                                F(14, 75)         =      12.23
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6381
                                                Root MSE          =      .2948



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .130005   .0440988     2.95   0.004     .0421556    .2178543
gs_strictn~t |   .1140357   .0405742     2.81   0.006     .0332078    .1948637
semh_ba~2016 |  -.0032526   .0018254    -1.78   0.079     -.006889    .0003838
         ks2 |   .0784229   .0298481     2.63   0.010     .0189625    .1378833
         fsm |  -.0085672   .0036089    -2.37   0.020    -.0157565    -.001378
         eal |   .0131879   .0017373     7.59   0.000     .0097271    .0166487
         sen |   .0117575   .0055707     2.11   0.038     .0006601    .0228549
    log_size |  -.0575124   .1053424    -0.55   0.587    -.2673652    .1523404
years_sinc~d |  -.0157298   .0100463    -1.57   0.122    -.0357431    .0042836
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1418192    .040376     3.51   0.001     .0613861    .2222523
gs_strictn~t |   .0821772   .0433788     1.89   0.062    -.0042379    .1685923
semh_ba~2016 |  -.0049665   .0015064    -3.30   0.001    -.0079674   -.0019656
         ks2 |   .0935719   .0283611     3.30   0.001     .0370737    .1500702
         fsm |   -.007582   .0043318    -1.75   0.084    -.0162114    .0010475
         eal |   .0171555   .0020639     8.31   0.000      .013044    .0212671
         sen |   .0133476   .0057425     2.32   0.023     .0019079    .0247873
    log_size |  -.1431527   .1041584    -1.37   0.173    -.3506469    .0643415
years_sinc~d |  -.0149383   .0113126    -1.32   0.191    -.0374742    .0075975
     academy |

SEMH control — English (n=90): β_W= 0.142 β_S= 0.082

Linear regression                               Number of obs     =         90
                                                F(13, 75)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.4876
                                                Root MSE          =     .34055



------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1010183   .0603035     1.68   0.098    -.0191125    .2211491
gs_strictn~t |   .0859406   .0508016     1.69   0.095    -.0152615    .1871427
semh_ba~2016 |  -.0029189   .0016109    -1.81   0.074     -.006128    .0002902
         ks2 |    .039479   .0313185     1.26   0.211    -.0229105    .1018686
         fsm |  -.0103286   .0038492    -2.68   0.009    -.0179965   -.0026607
         eal |    .012586   .0021145     5.95   0.000     .0083737    .0167984
         sen |   .0139574   .0060118     2.32   0.023     .0019813    .0259335
    log_size |  -.0154142   .1238343    -0.12   0.901    -.2621048    .2312765
years_sinc~d |  -.0158567   .0113449    -1.40   0.166    -.0384569    .0067435
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1388857   .0590588     2.35   0.021     .0212345    .2565369
gs_strictn~t |   .1569558   .0517335     3.03   0.003     .0538974    .2600141
semh_ba~2016 |  -.0025916   .0025003    -1.04   0.303    -.0075724    .0023892
         ks2 |   .0910262   .0408216     2.23   0.029     .0097054     .172347
         fsm |   -.008791   .0045857    -1.92   0.059    -.0179262    .0003442
         eal |   .0144965   .0022231     6.52   0.000     .0100678    .0189251
         sen |   .0117004   .0071715     1.63   0.107     -.002586    .0259867
    log_size |  -.1100536    .141761    -0.78   0.440     -.392456    .1723487
years_sinc~d |  -.0131425   .0121558    -1.08   0.283     -.037358     .011073
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .136761   .0534089     2.56   0.012     .0303651    .2431569
gs_strictn~t |   .1130088   .0573342     1.97   0.052    -.0012069    .2272245
semh_ba~2016 |  -.0031408   .0024344    -1.29   0.201    -.0079904    .0017088
         ks2 |   .0824568   .0272924     3.02   0.003     .0280875    .1368262
         fsm |  -.0079437   .0044868    -1.77   0.081    -.0168819    .0009946
         eal |   .0100029   .0031601     3.17   0.002     .0037077    .0162981
         sen |   .0087296   .0068662     1.27   0.208    -.0049487    .0224078
    log_size |   .0210039   .1229898     0.17   0.865    -.2240043    .2660122
years_sinc~d |  -.0190314   .0105874    -1.80   0.076    -.0401225    .0020598
     academy |

In [16]:
* ================================================================
* Export: tab_robustness_overall.tex + tab_robustness_eng.tex
* Warmth and Strictness coefficients across robustness specs
* ================================================================

* Re-run primary (with Ofsted grade controls) alongside robustness specs
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store primary_`lbl'
}

* Overall P8 robustness table
esttab primary_Overall rob_ngrade_Overall rob_2324_Overall rob_att8_Overall ///
       rob_inter_Overall rob_semh_Overall ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_robustness_overall.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "\$W\$" gs_strictness_visit "\$S\$") ///
    mtitles("Primary" "No grade" "2023-24" "Att8 2425" "W\$\times\$S" "SEMH ctrl") ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Robustness: Overall Progress 8")

* English P8 robustness table
esttab primary_English rob_ngrade_English rob_2324_English rob_att8_English ///
       rob_inter_English rob_semh_English ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_robustness_eng.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "\$W\$" gs_strictness_visit "\$S\$") ///
    mtitles("Primary" "No grade" "2023-24" "Att8 2425" "W\$\times\$S" "SEMH ctrl") ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Robustness: English P8 component")

display "Robustness tables (Overall + English) written."


Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5952
                                                Root MSE          =     .32043



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1273558   .0450304     2.83   0.006     .0377594    .2169523
gs_strictn~t |   .1199549   .0472976     2.54   0.013     .0258474    .2140623
         ks2 |   .0571472   .0200136     2.86   0.005     .0173264    .0969679
         fsm |  -.0067931    .003953    -1.72   0.090    -.0146583    .0010721
         eal |   .0104332   .0017253     6.05   0.000     .0070005     .013866
         sen |   .0081699   .0057042     1.43   0.156    -.0031796    .0195194
    log_size |  -.0169044   .1065872    -0.16   0.874    -.2289795    .1951706
years_sinc~d |   -.008202    .010188    -0.81   0.423     -.028473    .0120689
     academy |   .0155712    .101014     0.15   0.878    -.1854149    .2165574
   urban_bin |


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =      10.03
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5057
                                                Root MSE          =      .3426

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0963812   .0579606     1.66   0.100    -.0189422    .2117045
gs_strictn~t |   .1068404   .0510776     2.09   0.040      .005212    .2084687
         ks2 |   .0394035   .0176788     2.23   0.029     .0042283    .0745788
         fsm |

  -.0088499   .0039605    -2.23   0.028      -.01673   -.0009698
         eal |   .0107847   .0017632     6.12   0.000     .0072766    .0142929
         sen |   .0105056   .0057126     1.84   0.070    -.0008607     .021872
    log_size |  -.0087769   .1184769    -0.07   0.941    -.2445088     .226955
years_sinc~d |  -.0093762    .010945    -0.86   0.394    -.0311533    .0124009
     academy |  -.0071375   .1130239    -0.06   0.950    -.2320197    .2177447
   urban_bin |     .06271   .1025265     0.61   0.542    -.1412855    .2667055
   selective |   .0312282   .3225837     0.10   0.923    -.6106122    .6730687
             |
ofsted_~2019 |
          3  |  -.2572682    .101089    -2.54   0.013    -.4584037   -.0561328
          4  |  -.2508771   .1496178    -1.68   0.097    -.5485695    .0468154
             |
       _cons |  -5.385078   2.330672    -2.31   0.023    -10.02238   -.7477708
------------------------------------------------------------------------------

Linear regression   

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1458399   .0599625     2.43   0.017     .0265334    .2651464
gs_strictn~t |   .1548066   .0578116     2.68   0.009     .0397796    .2698335
         ks2 |   .0592719   .0244213     2.43   0.017     .0106812    .1078625
         fsm |  -.0075709   .0046497    -1.63   0.107    -.0168224    .0016805
         eal |   .0122905   .0020765     5.92   0.000     .0081589    .0164221
         sen |   .0094793   .0072462     1.31   0.195    -.0049384     .023897
    log_size |  -.0469809   .1448128    -0.32   0.746    -.3351129    .2411511
years_sinc~d |  -.0072161    .012158    -0.59   0.554    -.0314066    .0169744
     academy |    .028049   .1175658     0.24   0.812    -.2058701    .2619681
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1243832   .0522875     2.38   0.020     .0203476    .2284189
gs_strictn~t |   .1225779   .0598856     2.05   0.044     .0034244    .2417314
         ks2 |   .0651215   .0212488     3.06   0.003      .022843       .1074
         fsm |   -.005527     .00473    -1.17   0.246    -.0149383    .0038842
         eal |   .0068437   .0027033     2.53   0.013      .001465    .0122225
         sen |   .0043785   .0069276     0.63   0.529    -.0094051    .0181622
    log_size |   .0602892   .1183337     0.51   0.612    -.1751578    .2957362
years_sinc~d |  -.0104628   .0112831    -0.93   0.357    -.0329127    .0119871
     academy |  -.0189095   .1167352    -0.16   0.872    -.2511759     .213357
   urban_bin |

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_robustness_overall.tex)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_robustness_eng.tex)
Robustness tables (Overall + English) written.


In [17]:
* ================================================================
* ENACTED vs ESPOUSED: Interview-only scores on 303 schools
* Uses gs_w3_adj (warmth interview) and mean(gs_s3, gs_s4) (strictness interview)
* Expected: lower coefficients than visit-only (espoused overclaims warmth)
* ================================================================
estimates clear

* Create interview-only scores (0-10 scale)
gen warmth_interview_only  = gs_w3_adj * 2
gen strict_interview_only  = (gs_s3 + gs_s4) / 2 * 2  if !missing(gs_s3) & !missing(gs_s4)

destring warmth_interview_only strict_interview_only, replace force

count if tier2 & !missing(warmth_interview_only) & !missing(strict_interview_only)

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    * Interview-only scores, n=267-303 (Tier 2, with pre-COVID grade where available)
    regress `outcome' warmth_interview_only strict_interview_only $controls if tier2, vce(hc3)
    estimates store espoused_`lbl'
    display _newline "Espoused (interview-only, n=" e(N) ") — `lbl':"
    display "  β_W = " %6.3f _b[warmth_interview_only] ///
            "  (se=" %6.3f _se[warmth_interview_only] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[warmth_interview_only]/_se[warmth_interview_only])))
    display "  β_S = " %6.3f _b[strict_interview_only] ///
            "  (se=" %6.3f _se[strict_interview_only] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[strict_interview_only]/_se[strict_interview_only])))
    display "  R² = " %6.4f e(r2)
}

(3,029 missing values generated)
(3,029 missing values generated)
warmth_interview_only already numeric; no replace
strict_interview_only already numeric; no replace
  303



Linear regression                               Number of obs     =        267
                                                F(13, 253)        =      19.78
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4289
                                                Root MSE          =     .36571

------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0201603   .0144507     1.40   0.164    -.0082986    .0486192
strict_int~y |   .0610319   .0286608     2.13   0.034     .0045877    .1174761
         ks2 |   .0516683   .0099823     5.18   0.000     .0320094    .0713272
         fsm |  -.0111924   .0026326    -4.25   0.000    -.0163769   -.0060078
         eal

                                                F(13, 253)        =      12.73
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4079
                                                Root MSE          =     .37317

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0165382   .0158744     1.04   0.298    -.0147246     .047801
strict_int~y |   .0722256   .0287596     2.51   0.013     .0155868    .1288644
         ks2 |   .0480186   .0117431     4.09   0.000     .0248919    .0711452
         fsm |  -.0095488   .0028739    -3.32   0.001    -.0152087    -.003889
         eal |   .0114851   .0014226     8.07   0.000     .0086834    .0142868
         sen 


Linear regression                               Number of obs     =        267
                                                F(13, 253)        =      12.41
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3558
                                                Root MSE          =     .37805

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |    .016085   .0147601     1.09   0.277    -.0129834    .0451534
strict_int~y |   .0267466   .0288967     0.93   0.356    -.0301622    .0836553
         ks2 |   .0409098    .010499     3.90   0.000     .0202332    .0615863
         fsm |  -.0111835   .0027686    -4.04   0.000    -.0166359   -.0057312
         eal

  -4.331365   1.417848    -3.05   0.002    -7.123653   -1.539077
------------------------------------------------------------------------------

Espoused (interview-only, n=267) — Maths:
 β_W =  0.016 (se= 0.015) p=0.277
 β_S =  0.027 (se= 0.029) p=0.356
 R² = 0.3558

Linear regression                               Number of obs     =        267
                                                F(13, 253)        =      18.88
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4401
                                                Root MSE          =     .42612



------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0257814   .0165261     1.56   0.120    -.0067649    .0583277
strict_int~y |   .0744001     .03562     2.09   0.038     .0042507    .1445496
         ks2 |   .0564107   .0115335     4.89   0.000     .0336968    .0791246
         fsm |  -.0149505   .0030589    -4.89   0.000    -.0209747   -.0089262
         eal |   .0129162   .0015593     8.28   0.000     .0098453    .0159871
         sen |  -.0019526   .0042196    -0.46   0.644    -.0102626    .0063573
    log_size |   .0099217   .0885034     0.11   0.911    -.1643756    .1842189
years_sinc~d |   .0054357    .006701     0.81   0.418    -.0077611    .0186325
     academy |   .0535077   .0635816     0.84   0.401     -.071709    .1787244
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0212802   .0185367     1.15   0.252    -.0152258    .0577861
strict_int~y |   .0660398   .0367782     1.80   0.074    -.0063907    .1384702
         ks2 |   .0561088   .0124744     4.50   0.000     .0315419    .0806756
         fsm |  -.0090422   .0030864    -2.93   0.004    -.0151205   -.0029639
         eal |   .0067708   .0014707     4.60   0.000     .0038744    .0096671
         sen |  -.0064698    .004638    -1.39   0.164    -.0156038    .0026641
    log_size |   .0437315   .0845406     0.52   0.605    -.1227615    .2102245
years_sinc~d |   .0112416   .0069116     1.63   0.105    -.0023699    .0248531
     academy |   .0099576   .0770635     0.13   0.897    -.1418101    .1617253
   urban_bin |

In [18]:
* ================================================================
* Enacted vs Espoused comparison: same 102 schools, visit-only vs interview-only
* More direct comparison — holds sample constant
* ================================================================
estimates clear

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    * Visit-only (enacted)
    regress `outcome' gs_warmth_visit gs_strictness_visit $controls if tier1, vce(hc3)
    estimates store enacted_`lbl'
    
    * Interview-only (espoused)
    regress `outcome' warmth_interview_only strict_interview_only $controls if tier1, vce(hc3)
    estimates store espoused102_`lbl'
    
    display "Enacted vs Espoused on 102 schools — `lbl':"
    estimates restore enacted_`lbl'
    display "  Enacted:  β_W=" %6.3f _b[gs_warmth_visit] "  β_S=" %6.3f _b[gs_strictness_visit]
    estimates restore espoused102_`lbl'
    display "  Espoused: β_W=" %6.3f _b[warmth_interview_only] "  β_S=" %6.3f _b[strict_interview_only]
}

* Export enacted vs espoused table (4 component outcomes — Overall excluded to keep table width)
esttab enacted_English espoused102_English enacted_Maths espoused102_Maths ///
       enacted_EBaC espoused102_EBaC enacted_Open espoused102_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_enacted_espoused.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit warmth_interview_only strict_interview_only) ///
    coeflabels(gs_warmth_visit "Warmth (visit, \$W_{12}\$)" ///
               gs_strictness_visit "Strictness (visit, \$S_{12}\$)" ///
               warmth_interview_only "Warmth (interview, \$W_3\$)" ///
               strict_interview_only "Strictness (interview, \$\bar{S}_{34}\$)") ///
    mtitles("Enact" "Espo" "Enact" "Espo" "Enact" "Espo" "Enact" "Espo") ///
    mgroups("English" "Maths" "EBaC" "Open", ///
            pattern(1 0 1 0 1 0 1 0) prefix(\multicolumn{2}{c}{) suffix(})) ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) nonumbers ///
    title("Enacted vs. espoused culture scores as predictors (\$N=102\$)")

display "tab_enacted_espoused.tex written."


Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5952
                                                Root MSE          =     .32043



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1273558   .0450304     2.83   0.006     .0377594    .2169523
gs_strictn~t |   .1199549   .0472976     2.54   0.013     .0258474    .2140623
         ks2 |   .0571472   .0200136     2.86   0.005     .0173264    .0969679
         fsm |  -.0067931    .003953    -1.72   0.090    -.0146583    .0010721
         eal |   .0104332   .0017253     6.05   0.000     .0070005     .013866
         sen |   .0081699   .0057042     1.43   0.156    -.0031796    .0195194
    log_size |  -.0169044   .1065872    -0.16   0.874    -.2289795    .1951706
years_sinc~d |   -.008202    .010188    -0.81   0.423     -.028473    .0120689
     academy |   .0155712    .101014     0.15   0.878    -.1854149    .2165574
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0216123   .0335778     0.64   0.522    -.0451969    .0884216
strict_int~y |    .048246   .0485613     0.99   0.323    -.0483758    .1448679
         ks2 |   .0746124   .0208886     3.57   0.001     .0330507    .1161741
         fsm |  -.0050719   .0052016    -0.98   0.332    -.0154213    .0052776
         eal |   .0091007   .0022397     4.06   0.000     .0046444     .013557
         sen |   .0066338    .007118     0.93   0.354    -.0075288    .0207964
    log_size |    .016396   .1320944     0.12   0.902    -.2464305    .2792225
years_sinc~d |  -.0057073   .0129768    -0.44   0.661     -.031527    .0201125
     academy |  -.0150812   .1311499    -0.11   0.909    -.2760283    .2458659
   urban_bin |


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =       6.89
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5616
                                                Root MSE          =     .35866

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1417135    .043923     3.23   0.002     .0543205    .2291064
gs_strictn~t |   .0792188   .0542554     1.46   0.148    -.0287325      .18717
         ks2 |   .0587733   .0246862     2.38   0.020     .0096555    .1078911
         fsm |  -.0056789   .0048108    -1.18   0.241    -.0152509     .003893
         eal


          4  |  -.1396433   .2123683    -0.66   0.513    -.5621895    .2829029
             |
       _cons |  -7.020901   2.902094    -2.42   0.018    -12.79516   -1.246643
------------------------------------------------------------------------------

Linear regression                               Number of obs     =         95
                                                F(13, 81)         =       4.50
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4469
                                                Root MSE          =     .40287



...


Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5157
                                                Root MSE          =     .37151



------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1243832   .0522875     2.38   0.020     .0203476    .2284189
gs_strictn~t |   .1225779   .0598856     2.05   0.044     .0034244    .2417314
         ks2 |   .0651215   .0212488     3.06   0.003      .022843       .1074
         fsm |   -.005527     .00473    -1.17   0.246    -.0149383    .0038842
         eal |   .0068437   .0027033     2.53   0.013      .001465    .0122225
         sen |   .0043785   .0069276     0.63   0.529    -.0094051    .0181622
    log_size |   .0602892   .1183337     0.51   0.612    -.1751578    .2957362
years_sinc~d |  -.0104628   .0112831    -0.93   0.357    -.0329127    .0119871
     academy |  -.0189095   .1167352    -0.16   0.872    -.2511759     .213357
   urban_bin |


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =       5.99
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3775
                                                Root MSE          =     .42122

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   -.007345   .0391076    -0.19   0.851    -.0851569    .0704668
strict_int~y |   .0835572   .0540911     1.54   0.126    -.0240672    .1911816
         ks2 |   .0853845   .0221645     3.85   0.000     .0412841     .129485
         fsm |  -.0043144   .0058429    -0.74   0.462      -.01594    .0073112
         eal

  -9.925715   2.989448    -3.32   0.001    -15.87378   -3.977651
------------------------------------------------------------------------------
Enacted vs Espoused on 102 schools — Open:
(results enacted_Open are active now)
 Enacted: β_W= 0.124 β_S= 0.123
(results espoused102_Open are active now)
 Espoused: β_W=-0.007 β_S= 0.084


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_enacted_espoused.tex)
tab_enacted_espoused.tex written.


In [19]:
* ================================================================
* Exploratory: Warmth and strictness GAP as predictor
* (enacted - espoused): does cultural coherence predict outcomes?
* ================================================================
gen warmth_gap  = gs_warmth_visit - warmth_interview_only
gen strict_gap  = gs_strictness_visit - strict_interview_only

display "Warmth gap (V - I) distribution:"
summarize warmth_gap if tier1

display "Strictness gap (V - I) distribution:"
summarize strict_gap if tier1

* Quick correlations with outcomes
display _newline "Correlation: warmth gap vs P8 outcomes:"
correlate warmth_gap strict_gap p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg if tier1

(3,230 missing values generated)
(3,230 missing values generated)
Warmth gap (V - I) distribution:

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
  warmth_gap |        102   -.4490694    1.506796       -3.5   4.094445
Strictness gap (V - I) distribution:

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
  strict_gap |        102   -.0074375    1.186797      -2.45        3.5

Correlation: warmth gap vs P8 outcomes:
(obs=102)

             | warmth~p strict~p p8~g_avg p8meam~g p8~c_avg p8meao~g
-------------+------------------------------------------------------
  warmth_gap |   1.0000
  strict_gap |   0.2601   1.0000
p8meaeng_avg |   0.0365   0.1737   1.0000
p8meamat_avg |  -0.0109   0.3124   0.8142   1.0000
p8meaebac_~g |  -0.0061   0.2602   0.8966   0.8636   1.0000
p8meaopen_~g |   0.0605   0.20

In [20]:
* ================================================================
* Export: tab_continuity_robustness.tex
* Primary (n≈95) vs continuity-restricted (unchanged HT, n≈23)
* ================================================================

* Re-run primary for comparison (components only — small n makes Overall redundant)
estimates clear
foreach outcome in p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8meaeng_avg", "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open")))
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store primary2_`lbl'
    regress `outcome' $WS $controls_ngrade if tier1 & ht_changed==0, vce(hc3)
    estimates store cont2_`lbl'
}

esttab primary2_English cont2_English primary2_Maths cont2_Maths ///
       primary2_EBaC cont2_EBaC primary2_Open cont2_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_continuity_robustness.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "Warmth (\$W\$)" gs_strictness_visit "Strictness (\$S\$)") ///
    mtitles("Full" "Cont" "Full" "Cont" "Full" "Cont" "Full" "Cont") ///
    mgroups("English" "Maths" "EBaC" "Open", ///
            pattern(1 0 1 0 1 0 1 0) prefix(\multicolumn{2}{c}{) suffix(})) ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) nonumbers ///
    title("Headteacher continuity robustness (Full sample vs. unchanged HT)")

display "tab_continuity_robustness.tex written."


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =       6.89
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5616
                                                Root MSE          =     .35866



------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1417135    .043923     3.23   0.002     .0543205    .2291064
gs_strictn~t |   .0792188   .0542554     1.46   0.148    -.0287325      .18717
         ks2 |   .0587733   .0246862     2.38   0.020     .0096555    .1078911
         fsm |  -.0056789   .0048108    -1.18   0.241    -.0152509     .003893
         eal |   .0131023   .0021442     6.11   0.000     .0088361    .0173686
         sen |   .0085504   .0060538     1.41   0.162    -.0034946    .0205955
    log_size |  -.0965717   .1113335    -0.87   0.388    -.3180905    .1249472
years_sinc~d |   -.005403   .0112837    -0.48   0.633    -.0278541    .0170481
     academy |   .0708777   .1126417     0.63   0.531     -.153244    .2949993
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    -.04871   .2063203    -0.24   0.818     -.502818    .4053979
gs_strictn~t |   .3327074   .2784601     1.19   0.257    -.2801792     .945594
         ks2 |   .1045867   .1311403     0.80   0.442    -.1840512    .3932245
         fsm |  -.0062608   .0189284    -0.33   0.747     -.047922    .0354004
         eal |   .0191271   .0082976     2.31   0.042     .0008642      .03739
         sen |   .0300736   .0461152     0.65   0.528    -.0714253    .1315726
    log_size |  -.0574057   .8134815    -0.07   0.945    -1.847866    1.733055
years_sinc~d |  -.0064898   .0400615    -0.16   0.874    -.0946645     .081685
     academy |   .1542103   .5272794     0.29   0.775    -1.006324    1.314744
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0963812   .0579606     1.66   0.100    -.0189422    .2117045
gs_strictn~t |   .1068404   .0510776     2.09   0.040      .005212    .2084687
         ks2 |   .0394035   .0176788     2.23   0.029     .0042283    .0745788
         fsm |  -.0088499   .0039605    -2.23   0.028      -.01673   -.0009698
         eal |   .0107847   .0017632     6.12   0.000     .0072766    .0142929
         sen |   .0105056   .0057126     1.84   0.070    -.0008607     .021872
    log_size |  -.0087769   .1184769    -0.07   0.941    -.2445088     .226955
years_sinc~d |  -.0093762    .010945    -0.86   0.394    -.0311533    .0124009
     academy |  -.0071375   .1130239    -0.06   0.950    -.2320197    .2177447
   urban_bin |


Linear regression                               Number of obs     =         23
                                                F(11, 11)         =       0.72
                                                Prob > F          =     0.7006
                                                R-squared         =     0.5291
                                                Root MSE          =     .35823

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0546979   .1680987    -0.33   0.751    -.4246806    .3152849
gs_strictn~t |   .2072467   .2198563     0.94   0.366    -.2766537     .691147
         ks2 |  -.0380629   .0977097    -0.39   0.704    -.2531206    .1769947
         fsm |  -.0134195     .01695    -0.79   0.445    -.0507262    .0238872
         eal

...

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1458399   .0599625     2.43   0.017     .0265334    .2651464
gs_strictn~t |   .1548066   .0578116     2.68   0.009     .0397796    .2698335
         ks2 |   .0592719   .0244213     2.43   0.017     .0106812    .1078625
         fsm |  -.0075709   .0046497    -1.63   0.107    -.0168224    .0016805
         eal |   .0122905   .0020765     5.92   0.000     .0081589    .0164221
         sen |   .0094793   .0072462     1.31   0.195    -.0049384     .023897
    log_size |  -.0469809   .1448128    -0.32   0.746    -.3351129    .2411511
years_sinc~d |  -.0072161    .012158    -0.59   0.554    -.0314066    .0169744
     academy |    .028049   .1175658     0.24   0.812    -.2058701    .2619681
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0483485   .2338107    -0.21   0.840    -.5629624    .4662654
gs_strictn~t |    .242196   .3241881     0.75   0.471    -.4713373    .9557293
         ks2 |   .0768108   .1526645     0.50   0.625    -.2592015    .4128231
         fsm |  -.0063531   .0192433    -0.33   0.747    -.0487074    .0360011
         eal |   .0111665   .0115284     0.97   0.354    -.0142073    .0365403
         sen |   .0208881   .0424472     0.49   0.632    -.0725376    .1143138
    log_size |  -.1899785   1.001828    -0.19   0.853    -2.394988    2.015031
years_sinc~d |   .0000444   .0474141     0.00   0.999    -.1043133    .1044022
     academy |   .2199353   .6760508     0.33   0.751    -1.268042    1.707913
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1243832   .0522875     2.38   0.020     .0203476    .2284189
gs_strictn~t |   .1225779   .0598856     2.05   0.044     .0034244    .2417314
         ks2 |   .0651215   .0212488     3.06   0.003      .022843       .1074
         fsm |   -.005527     .00473    -1.17   0.246    -.0149383    .0038842
         eal |   .0068437   .0027033     2.53   0.013      .001465    .0122225
         sen |   .0043785   .0069276     0.63   0.529    -.0094051    .0181622
    log_size |   .0602892   .1183337     0.51   0.612    -.1751578    .2957362
years_sinc~d |  -.0104628   .0112831    -0.93   0.357    -.0329127    .0119871
     academy |  -.0189095   .1167352    -0.16   0.872    -.2511759     .213357
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0495255   .2520901     0.20   0.848    -.5053211    .6043721
gs_strictn~t |   .2856897   .2625869     1.09   0.300    -.2922602    .8636395
         ks2 |   .0414306   .1187619     0.35   0.734    -.2199626    .3028237
         fsm |   -.013205   .0162725    -0.81   0.434    -.0490204    .0226105
         eal |   .0178234   .0062481     2.85   0.016     .0040714    .0315754
         sen |  -.0050868   .0444961    -0.11   0.911     -.103022    .0928485
    log_size |  -.7112985    .823759    -0.86   0.406     -2.52438    1.101783
years_sinc~d |  -.0084775   .0464649    -0.18   0.859    -.1107461    .0937911
     academy |   .0272256   .5544047     0.05   0.962    -1.193011    1.247462
   urban_bin |

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_continuity_robustness.tex)
tab_continuity_robustness.tex written.


In [21]:
* ================================================================
* NATIONAL EXTENSION (E3): Ofsted LLM strictness -> P8
* N ~3,194 schools with ofsted_llmstrictnessscore, p8mea_avg, and controls
* ofsted_llmstrictnessscore on 1-5 scale; gs_strictness_visit on 0-10
* Warmth omitted: no valid national enacted warmth source
* No Ofsted grade control (score derived from the same Ofsted report)
* ================================================================

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))

    regress `outcome' ofsted_llmstrictnessscore $controls_ngrade, vce(hc3)
    estimates store nat_`lbl'
    display _newline "National (E3) -- `lbl' (n=" e(N) "):"
    display "  beta_S_ofsted = " %6.3f _b[ofsted_llmstrictnessscore] ///
            "  (se=" %6.3f _se[ofsted_llmstrictnessscore] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[ofsted_llmstrictnessscore]/_se[ofsted_llmstrictnessscore])))
    display "  R2 = " %6.4f e(r2)
}
display _newline "Note: Tier 1 visit-based beta_S=0.120 on 0-10 scale; multiply Ofsted beta by 2 for approx comparison."

esttab nat_Overall nat_English nat_Maths nat_EBaC nat_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_national_strictness.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(ofsted_llmstrictnessscore) ///
    coeflabels(ofsted_llmstrictnessscore "Strictness (Ofsted LLM, 1--5 scale)") ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("National extension: Ofsted LLM strictness and Progress~8 ($N \approx 3{,}194$)")

display "tab_national_strictness.tex written."



Linear regression                               Number of obs     =      3,148
                                                F(10, 3137)       =     308.37
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5060
                                                Root MSE          =     .35596



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |   .1350234   .0085915    15.72   0.000     .1181779    .1518689
         ks2 |   .0368824    .003344    11.03   0.000     .0303256    .0434391
         fsm |  -.0126478    .000858   -14.74   0.000      -.01433   -.0109656
         eal |   .0095426   .0005456    17.49   0.000     .0084728    .0106123
         sen |  -.0018614   .0012783    -1.46   0.145    -.0043678     .000645
    log_size |   .1255602   .0221589     5.67   0.000     .0821127    .1690077
years_sinc~d |   .0080933   .0016838     4.81   0.000     .0047918    .0113948
     academy |   .0314522   .0168719     1.86   0.062    -.0016289    .0645333
   urban_bin |  -.0675556   .0234873    -2.88   0.004    -.1136077   -.0215035
   selective |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |   .1235215   .0092314    13.38   0.000     .1054213    .1416217
         ks2 |   .0377901   .0036761    10.28   0.000     .0305823    .0449978
         fsm |  -.0104612   .0009294   -11.26   0.000    -.0122835   -.0086389
         eal |   .0105135   .0005986    17.56   0.000     .0093398    .0116872
         sen |  -.0031166   .0013534    -2.30   0.021    -.0057703   -.0004629
    log_size |    .085487   .0239273     3.57   0.000     .0385722    .1324017
years_sinc~d |   .0093505   .0018726     4.99   0.000     .0056788    .0130221
     academy |   .0289336   .0183999     1.57   0.116    -.0071434    .0650107
   urban_bin |  -.0543712   .0252675    -2.15   0.031    -.1039138   -.0048287
   selective |


Linear regression                               Number of obs     =      3,148
                                                F(10, 3137)       =     241.27
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4595
                                                Root MSE          =      .3567

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |   .1169728   .0085503    13.68   0.000      .100208    .1337376
         ks2 |   .0220161   .0035088     6.27   0.000     .0151363    .0288959
         fsm |   -.014654   .0007967   -18.39   0.000     -.016216   -.0130919
         eal |   .0096204   .0004886    19.69   0.000     .0086623    .0105785
         sen


Linear regression                               Number of obs     =      3,148
                                                F(10, 3137)       =     299.04
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4852
                                                Root MSE          =     .41864

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |   .1365141   .0102926    13.26   0.000     .1163331    .1566951
         ks2 |   .0394973   .0039496    10.00   0.000     .0317532    .0472415
         fsm |  -.0162532   .0010826   -15.01   0.000    -.0183758   -.0141305
         eal |   .0113592     .00069    16.46   0.000     .0100064     .012712
         sen


Linear regression                               Number of obs     =      3,148
                                                F(10, 3137)       =     207.57
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4100
                                                Root MSE          =     .43624



------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |    .158012     .01006    15.71   0.000     .1382872    .1777368
         ks2 |   .0419267   .0038935    10.77   0.000     .0342927    .0495607
         fsm |  -.0095369   .0010314    -9.25   0.000    -.0115591   -.0075147
         eal |   .0070687    .000679    10.41   0.000     .0057373    .0084001
         sen |  -.0027875   .0015405    -1.81   0.070    -.0058081     .000233
    log_size |    .190477   .0243442     7.82   0.000     .1427448    .2382093
years_sinc~d |   .0107345   .0020951     5.12   0.000     .0066267    .0148424
     academy |    .051167   .0205218     2.49   0.013     .0109294    .0914046
   urban_bin |  -.0456922    .029028    -1.57   0.116     -.102608    .0112236
   selective |

(file C:/Users/damia/OneDrive/Documents/Schools
    Project/thesis/tables/tab_national_strictness.tex not found)
(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_national_strictness.tex)


tab_national_strictness.tex written.


In [22]:
* ================================================================
* SCORE SENSITIVITY (G2b + G3)
* G2b: visit-only (primary) vs composite-current (60/40) vs composite-v1
*   warmth formula is IDENTICAL in v1 and current (gs_warmth_score_v1 = gs_warmth_composite)
*   strictness differs: v1 uses S4 iq-adjusted; current uses S4 raw (~0.22 pts higher in T1)
* G3: 60/40 (current) vs 50/50 visit/interview weighting
* ================================================================

* --- G2b composite current vs v1 ---
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))

    regress `outcome' gs_warmth_composite gs_strictness_composite $controls if tier1, vce(hc3)
    estimates store comp_curr_`lbl'

    regress `outcome' gs_warmth_score_v1 gs_strictness_score_v1 $controls if tier1, vce(hc3)
    estimates store comp_v1_`lbl'
}

* --- G3: 50/50 weighting ---
* warmth_5050 = 0.5*visit_W(0-10) + 0.5*W3_adj(0-5)*2
* strictness_5050 = 0.5*visit_S(0-10) + 0.5*(S3+S4)(each 0-5, sum 0-10)
cap drop warmth_5050 strictness_5050
gen warmth_5050     = 0.5 * gs_warmth_visit     + 0.5 * gs_w3_adj * 2
gen strictness_5050 = 0.5 * gs_strictness_visit + 0.5 * (gs_s3 + gs_s4)
label var warmth_5050     "Warmth composite (50/50 weighting)"
label var strictness_5050 "Strictness composite (50/50 weighting)"

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    regress `outcome' warmth_5050 strictness_5050 $controls if tier1, vce(hc3)
    estimates store w5050_`lbl'
    display "50/50 -- `lbl' (n=" e(N) "): beta_W=" %6.3f _b[warmth_5050] " beta_S=" %6.3f _b[strictness_5050]
}

* Re-run visit-only for side-by-side export
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store visit_`lbl'
}

esttab visit_Overall comp_curr_Overall comp_v1_Overall w5050_Overall ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_score_sensitivity.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit gs_warmth_composite gs_strictness_composite ///
         gs_warmth_score_v1 gs_strictness_score_v1 warmth_5050 strictness_5050) ///
    coeflabels(gs_warmth_visit         "Warmth (visit-only, primary)" ///
               gs_strictness_visit     "Strictness (visit-only, primary)" ///
               gs_warmth_composite     "Warmth (composite 60/40, current)" ///
               gs_strictness_composite "Strictness (composite 60/40, current)" ///
               gs_warmth_score_v1      "Warmth (composite 60/40, v1)" ///
               gs_strictness_score_v1  "Strictness (composite 60/40, v1)" ///
               warmth_5050             "Warmth (composite 50/50)" ///
               strictness_5050         "Strictness (composite 50/50)") ///
    mtitles("Visit-only" "Composite 60/40" "Composite v1" "Composite 50/50") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Score construction sensitivity: Overall Progress~8 (Tier 1, $n \approx 95$)")

display "tab_score_sensitivity.tex written."



Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5536
                                                Root MSE          =     .33648



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~e |    .112172   .0558541     2.01   0.048     .0010398    .2233041
gs_strictn~e |    .155401   .0629311     2.47   0.016     .0301879    .2806141
         ks2 |   .0668441   .0198504     3.37   0.001     .0273481    .1063402
         fsm |  -.0062979   .0044067    -1.43   0.157    -.0150657      .00247
         eal |   .0098088   .0019679     4.98   0.000     .0058932    .0137244
         sen |   .0088388   .0061391     1.44   0.154     -.003376    .0210537
    log_size |    .011171   .1024667     0.11   0.913    -.1927056    .2150476
years_sinc~d |  -.0035664   .0114639    -0.31   0.757    -.0263758    .0192431
     academy |  -.0055666   .1089696    -0.05   0.959     -.222382    .2112487
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~1 |   .1104568   .0637101     1.73   0.087    -.0163063      .23722
gs_strictn~1 |   .1227853   .0670826     1.83   0.071    -.0106881    .2562587
         ks2 |   .0681334   .0200278     3.40   0.001     .0282844    .1079823
         fsm |  -.0062755    .004532    -1.38   0.170    -.0152927    .0027417
         eal |   .0098737   .0020717     4.77   0.000     .0057517    .0139957
         sen |   .0090354   .0063424     1.42   0.158     -.003584    .0216549
    log_size |   .0099656   .1076969     0.09   0.927    -.2043174    .2242486
years_sinc~d |  -.0037857   .0117144    -0.32   0.747    -.0270938    .0195223
     academy |  -.0003891   .1115998    -0.00   0.997    -.2224377    .2216596
   urban_bin |


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =       5.71
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5312
                                                Root MSE          =     .37087

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~e |   .1075427    .064804     1.66   0.101    -.0213969    .2364822
gs_strictn~e |   .1337212   .0714599     1.87   0.065    -.0084616    .2759041
         ks2 |    .066821   .0243283     2.75   0.007     .0184154    .1152266
         fsm |  -.0049456   .0052465    -0.94   0.349    -.0153845    .0054934
         eal


Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5186
                                                Root MSE          =     .37583

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~1 |   .1090068   .0709283     1.54   0.128    -.0321182    .2501318
gs_strictn~1 |   .1008329     .07394     1.36   0.176    -.0462844    .2479502
         ks2 |   .0678348   .0241825     2.81   0.006     .0197193    .1159504
         fsm |  -.0049122   .0053734    -0.91   0.363    -.0156036    .0057792
         eal

...


Linear regression                               Number of obs     =         95
                                                F(13, 81)         =      10.03
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5057
                                                Root MSE          =      .3426

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0963812   .0579606     1.66   0.100    -.0189422    .2117045
gs_strictn~t |   .1068404   .0510776     2.09   0.040      .005212    .2084687
         ks2 |   .0394035   .0176788     2.23   0.029     .0042283    .0745788
         fsm |  -.0088499   .0039605    -2.23   0.028      -.01673   -.0009698
         eal

  -.0093762    .010945    -0.86   0.394    -.0311533    .0124009
     academy |  -.0071375   .1130239    -0.06   0.950    -.2320197    .2177447
   urban_bin |     .06271   .1025265     0.61   0.542    -.1412855    .2667055
   selective |   .0312282   .3225837     0.10   0.923    -.6106122    .6730687
             |
ofsted_~2019 |
          3  |  -.2572682    .101089    -2.54   0.013    -.4584037   -.0561328
          4  |  -.2508771   .1496178    -1.68   0.097    -.5485695    .0468154
             |
       _cons |  -5.385078   2.330672    -2.31   0.023    -10.02238   -.7477708
------------------------------------------------------------------------------

Linear regression                               Number of obs     =         95
                                                F(12, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5704
                    

gs_strictn~t |   .1548066   .0578116     2.68   0.009     .0397796    .2698335
         ks2 |   .0592719   .0244213     2.43   0.017     .0106812    .1078625
         fsm |  -.0075709   .0046497    -1.63   0.107    -.0168224    .0016805
         eal |   .0122905   .0020765     5.92   0.000     .0081589    .0164221
         sen |   .0094793   .0072462     1.31   0.195    -.0049384     .023897
    log_size |  -.0469809   .1448128    -0.32   0.746    -.3351129    .2411511
years_sinc~d |  -.0072161    .012158    -0.59   0.554    -.0314066    .0169744
     academy |    .028049   .1175658     0.24   0.812    -.2058701    .2619681
   urban_bin |   .0472236   .1498146     0.32   0.753    -.2508604    .3453076
   selective |   .1171025   .3693405     0.32   0.752    -.6177692    .8519743
             |
ofsted_~2019 |
          3  |  -.2852205   .1357283    -2.10   0.039    -.5552771   -.0151638
          4  |  -.2591303    .135858    -1.91   0.060    -.5294452    .0111845
             |
       

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1243832   .0522875     2.38   0.020     .0203476    .2284189
gs_strictn~t |   .1225779   .0598856     2.05   0.044     .0034244    .2417314
         ks2 |   .0651215   .0212488     3.06   0.003      .022843       .1074
         fsm |   -.005527     .00473    -1.17   0.246    -.0149383    .0038842
         eal |   .0068437   .0027033     2.53   0.013      .001465    .0122225
         sen |   .0043785   .0069276     0.63   0.529    -.0094051    .0181622
    log_size |   .0602892   .1183337     0.51   0.612    -.1751578    .2957362
years_sinc~d |  -.0104628   .0112831    -0.93   0.357    -.0329127    .0119871
     academy |  -.0189095   .1167352    -0.16   0.872    -.2511759     .213357
   urban_bin |

(file C:/Users/damia/OneDrive/Documents/Schools
    Project/thesis/tables/tab_score_sensitivity.tex not found)
(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_score_sensitivity.tex)
tab_score_sensitivity.tex written.


In [23]:
* ================================================================
* SEMH MECHANISM TEST (G7)
* H: strict schools accumulate lower SEMH share than baseline predicts
* Consistent with behavioural sorting/exclusion mechanism
* Tier 1: gs_strictness_visit; National: ofsted_llmstrictnessscore
* semh_baseline_2016 and semh_current are raw pupil COUNTS -- divide by size for shares
* ================================================================

cap drop semh_share_baseline semh_share_current
gen semh_share_baseline = semh_baseline_2016 / size * 100
gen semh_share_current  = semh_current       / size * 100
label var semh_share_baseline "SEMH prevalence 2015-16 (% of roll)"
label var semh_share_current  "SEMH prevalence 2023-24 (% of roll)"

display _newline "SEMH share distributions:"
summarize semh_share_baseline semh_share_current

* Spec 1: Tier 1 -- strictness (gold-standard visit)
regress semh_share_current gs_strictness_visit semh_share_baseline $controls_ngrade if tier1, vce(hc3)
estimates store semh_t1_s
display _newline "SEMH mechanism -- Tier 1 strictness (n=" e(N) "):"
display "  beta_S = " %6.3f _b[gs_strictness_visit] ///
        "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_strictness_visit]/_se[gs_strictness_visit])))

* Spec 2: Tier 1 -- warmth (gold-standard visit)
regress semh_share_current gs_warmth_visit semh_share_baseline $controls_ngrade if tier1, vce(hc3)
estimates store semh_t1_w
display _newline "SEMH mechanism -- Tier 1 warmth (n=" e(N) "):"
display "  beta_W = " %6.3f _b[gs_warmth_visit] ///
        "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_warmth_visit]/_se[gs_warmth_visit])))

* Spec 3: National -- Ofsted LLM strictness (large-N)
regress semh_share_current ofsted_llmstrictnessscore semh_share_baseline $controls_ngrade, vce(hc3)
estimates store semh_nat_s
display _newline "SEMH mechanism -- National Ofsted strictness (n=" e(N) "):"
display "  beta_S_ofsted = " %6.3f _b[ofsted_llmstrictnessscore] ///
        "  p=" %5.3f (2*ttail(e(df_r), abs(_b[ofsted_llmstrictnessscore]/_se[ofsted_llmstrictnessscore])))

esttab semh_t1_s semh_t1_w semh_nat_s ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_semh_mechanism.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_strictness_visit gs_warmth_visit ofsted_llmstrictnessscore semh_share_baseline) ///
    coeflabels(gs_strictness_visit       "Strictness ($S_{\text{visit}}$)" ///
               gs_warmth_visit           "Warmth ($W_{\text{visit}}$)" ///
               ofsted_llmstrictnessscore "Strictness (Ofsted LLM, 1--5)" ///
               semh_share_baseline       "SEMH prevalence 2015--16 (\%)") ///
    mtitles("Tier 1 (S)" "Tier 1 (W)" "National (S)") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("SEMH mechanism: culture and current SEMH composition conditional on baseline")

display "tab_semh_mechanism.tex written."


(325 missing values generated)
(7 missing values generated)

SEMH share distributions:

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
semh_share~e |      3,007    2.407793    2.567472          0    64.7541
semh_share~t |      3,325    4.146441    2.559118          0   32.72727

Linear regression                               Number of obs     =         94
                                                F(11, 82)         =       7.10
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5514
                                                Root MSE          =     1.4779



------------------------------------------------------------------------------
             |             Robust HC3
semh_share~t | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_strictn~t |  -.2632502   .3090314    -0.85   0.397    -.8780122    .3515118
semh_share~e |   .1646551   .0668483     2.46   0.016     .0316725    .2976377
         ks2 |  -.1121407   .0875152    -1.28   0.204    -.2862362    .0619549
         fsm |   .0307503   .0205246     1.50   0.138    -.0100796    .0715803
         eal |  -.0315162   .0089124    -3.54   0.001    -.0492459   -.0137865
         sen |   .1140572   .0408333     2.79   0.006     .0328268    .1952876
    log_size |  -1.213828    .820486    -1.48   0.143    -2.846036    .4183798
years_sinc~d |   .0130435   .0462232     0.28   0.779    -.0789093    .1049962
     academy |  -.1495775   .4905398    -0.30   0.761    -1.125417    .8262625
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
semh_share~t | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0415081   .2152944    -0.19   0.848    -.4697972    .3867809
semh_share~e |   .1792727   .0609673     2.94   0.004     .0579894    .3005561
         ks2 |   -.103411   .0861755    -1.20   0.234    -.2748416    .0680196
         fsm |   .0314511   .0211011     1.49   0.140    -.0105258     .073428
         eal |  -.0324472   .0102329    -3.17   0.002    -.0528037   -.0120906
         sen |   .1124181   .0387134     2.90   0.005     .0354048    .1894315
    log_size |  -1.181396   .8202247    -1.44   0.154    -2.813084    .4502923
years_sinc~d |   .0200915   .0431434     0.47   0.643    -.0657344    .1059174
     academy |  -.1198699    .487402    -0.25   0.806    -1.089468     .849728
   urban_bin |


Linear regression                               Number of obs     =      2,909
                                                F(11, 2897)       =     122.65
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4885
                                                Root MSE          =     1.7636

------------------------------------------------------------------------------
             |             Robust HC3
semh_share~t | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |  -.1322261   .0407921    -3.24   0.001    -.2122105   -.0522417
semh_share~e |   .1858987   .0215972     8.61   0.000     .1435513    .2282461
         ks2 |  -.0728504   .0166268    -4.38   0.000     -.105452   -.0402487
         fsm |   .0218135   .0038494     5.67   0.000     .0142656    .0293614
         eal

(file C:/Users/damia/OneDrive/Documents/Schools
    Project/thesis/tables/tab_semh_mechanism.tex not found)
(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_semh_mechanism.tex)
tab_semh_mechanism.tex written.


In [24]:
* ================================================================
* MANAGEMENT DISCOURSE (G4): trx_llmmanagementscore -> P8
* Extended tier: n ~287 schools with interview transcript LLM scores + P8 + controls
* All trx_llm scores on 1-5 scale (not 0-10 like visit-based gold standard)
* Filter directly on !missing(trx_llmmanagementscore) -- avoids redefining tier2
* ================================================================

count if !missing(trx_llmmanagementscore) & !missing(p8mea_avg) & !missing(ks2)
display _newline "Sample: schools with trx_llmmanagementscore above"

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))

    regress `outcome' trx_llmwarmthscore trx_llmstrictnessscore trx_llmmanagementscore ///
        $controls_ngrade if !missing(trx_llmmanagementscore), vce(hc3)
    estimates store mgmt_`lbl'
    display _newline "Management discourse -- `lbl' (n=" e(N) "):"
    display "  beta_W(trx)=" %6.3f _b[trx_llmwarmthscore] ///
            "  beta_S(trx)=" %6.3f _b[trx_llmstrictnessscore] ///
            "  beta_M(trx)=" %6.3f _b[trx_llmmanagementscore] ///
            "  p_M=" %5.3f (2*ttail(e(df_r), abs(_b[trx_llmmanagementscore]/_se[trx_llmmanagementscore])))
}

esttab mgmt_Overall mgmt_English mgmt_Maths mgmt_EBaC mgmt_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_management_discourse.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(trx_llmwarmthscore trx_llmstrictnessscore trx_llmmanagementscore) ///
    coeflabels(trx_llmwarmthscore      "Warmth (transcript LLM, $\tilde{W}$)" ///
               trx_llmstrictnessscore  "Strictness (transcript LLM, $\tilde{S}$)" ///
               trx_llmmanagementscore  "Management discourse (transcript LLM, $\tilde{M}$)") ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Management discourse as predictor: extended tier ($N \approx 287$)")

display "tab_management_discourse.tex written."

  287

Sample: schools with trx_llmmanagementscore above

Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      20.29
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4333
                                                Root MSE          =     .37587



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
trx_llmwa~re |   .0987747    .066866     1.48   0.141    -.0328725    .2304218
trx_llmst~re |   .0185525   .0685586     0.27   0.787    -.1164271    .1535321
trx_llmman~e |  -.0923898   .0729475    -1.27   0.206    -.2360104    .0512309
         ks2 |   .0531457   .0099006     5.37   0.000     .0336532    .0726383
         fsm |  -.0116711   .0027865    -4.19   0.000    -.0171572   -.0061851
         eal |   .0100019   .0013012     7.69   0.000     .0074401    .0125636
         sen |  -.0036095   .0033416    -1.08   0.281    -.0101884    .0029695
    log_size |   .0407542   .0656334     0.62   0.535    -.0884662    .1699746
years_sinc~d |   .0079595   .0054344     1.46   0.144    -.0027398    .0186588
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
trx_llmwa~re |   .1032042   .0703904     1.47   0.144     -.035382    .2417903
trx_llmst~re |  -.0127787    .064731    -0.20   0.844    -.1402226    .1146652
trx_llmman~e |  -.0386538   .0690423    -0.56   0.576    -.1745858    .0972783
         ks2 |   .0474588   .0117059     4.05   0.000      .024412    .0705056
         fsm |  -.0101638   .0029795    -3.41   0.001    -.0160299   -.0042977
         eal |    .011231   .0014676     7.65   0.000     .0083416    .0141204
         sen |  -.0036482   .0035588    -1.03   0.306     -.010655    .0033585
    log_size |   .0392159    .065106     0.60   0.547    -.0889664    .1673981
years_sinc~d |   .0070015   .0057751     1.21   0.226    -.0043687    .0183717
     academy |


Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      13.57
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3716
                                                Root MSE          =      .3789

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
trx_llmwa~re |   .0969518   .0670889     1.45   0.150    -.0351343    .2290378
trx_llmst~re |   .0313618    .060362     0.52   0.604    -.0874802    .1502038
trx_llmman~e |  -.0889955   .0756926    -1.18   0.241    -.2380208    .0600298
         ks2 |   .0449702   .0100153     4.49   0.000     .0252518    .0646886
         fsm


Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      18.98
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4455
                                                Root MSE          =     .43292

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
trx_llmwa~re |    .125817   .0800292     1.57   0.117    -.0317463    .2833803
trx_llmst~re |    .064022   .0712414     0.90   0.370    -.0762397    .2042837
trx_llmman~e |  -.1083924   .0787575    -1.38   0.170    -.2634519     .046667
         ks2 |   .0581396    .011279     5.15   0.000     .0359334    .0803459
         fsm


Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      16.16
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3259
                                                Root MSE          =     .46969

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------


trx_llmwa~re |   .0715942   .0740267     0.97   0.334    -.0741511    .2173395
trx_llmst~re |  -.0163874   .0930937    -0.18   0.860    -.1996724    .1668976
trx_llmman~e |  -.1143358   .0877747    -1.30   0.194    -.2871486     .058477
         ks2 |   .0570695   .0119559     4.77   0.000     .0335305    .0806086
         fsm |  -.0091398   .0032158    -2.84   0.005    -.0154711   -.0028085
         eal |   .0070675   .0015056     4.69   0.000     .0041033    .0100317
         sen |   -.006327   .0049094    -1.29   0.199    -.0159928    .0033388
    log_size |   .0628134   .0833925     0.75   0.452    -.1013715    .2269983
years_sinc~d |   .0135147   .0066491     2.03   0.043     .0004238    .0266056
     academy |  -.0016364   .0789115    -0.02   0.983    -.1569991    .1537263
   urban_bin |  -.1051054   .0785601    -1.34   0.182    -.2597762    .0495654
   selective |   .2589342   .1244899     2.08   0.038     .0138357    .5040327
       _cons |  -6.142639    1.65618    -3.71   0.00

(file C:/Users/damia/OneDrive/Documents/Schools
    Project/thesis/tables/tab_management_discourse.tex not found)
(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_management_discourse.tex)
tab_management_discourse.tex written.


In [25]:
* ================================================================
* TEACHING PHILOSOPHY (national): web_id_llmteachingphilosophy -> P8
* 3 categories (v3 rubric): traditional (111), progressive (34), unmarked base (3,166)
* web_id_llmteachingphilosophy is a string variable (lowercase after case(lower))
* ================================================================

cap drop trad prog
gen trad = (web_id_llmteachingphilosophy == "traditional") if !missing(web_id_llmteachingphilosophy)
gen prog = (web_id_llmteachingphilosophy == "progressive") if !missing(web_id_llmteachingphilosophy)
label var trad "Traditional teaching philosophy (vs unmarked)"
label var prog "Progressive teaching philosophy (vs unmarked)"

display _newline "Category counts:"
count if trad == 1
count if prog == 1
count if !missing(web_id_llmteachingphilosophy) & !missing(p8mea_avg) & !missing(ks2)

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))

    regress `outcome' trad prog $controls_ngrade, vce(hc3)
    estimates store tp_`lbl'
    display _newline "Teaching philosophy -- `lbl' (n=" e(N) "):"
    display "  beta_trad=" %6.3f _b[trad] "  p=" %5.3f (2*ttail(e(df_r), abs(_b[trad]/_se[trad])))
    display "  beta_prog=" %6.3f _b[prog] "  p=" %5.3f (2*ttail(e(df_r), abs(_b[prog]/_se[prog])))
}

esttab tp_Overall tp_English tp_Maths tp_EBaC tp_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_teaching_philosophy.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(trad prog) ///
    coeflabels(trad "Traditional (vs unmarked)" prog "Progressive (vs unmarked)") ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Website teaching philosophy and Progress~8 (national, $N \approx 3{,}219$)")

display "tab_teaching_philosophy.tex written."


(21 missing values generated)
(21 missing values generated)

Category counts:
  472
  179
  3,219

Linear regression                               Number of obs     =      3,170
                                                F(11, 3158)       =     221.52
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4599
                                                Root MSE          =     .37272



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |    .038683   .0191832     2.02   0.044     .0010703    .0762958
        prog |  -.1030114   .0385377    -2.67   0.008    -.1785729   -.0274499
         ks2 |   .0465707   .0034391    13.54   0.000     .0398276    .0533138
         fsm |  -.0137072   .0008825   -15.53   0.000    -.0154374   -.0119769
         eal |   .0107137   .0005502    19.47   0.000     .0096348    .0117926
         sen |  -.0016457   .0013223    -1.24   0.213    -.0042384     .000947
    log_size |   .1298538   .0219938     5.90   0.000     .0867302    .1729774
years_sinc~d |   .0118072   .0017884     6.60   0.000     .0083006    .0153138
     academy |   .0162246   .0173123     0.94   0.349      -.01772    .0501691
   urban_bin |


Linear regression                               Number of obs     =      3,170
                                                F(11, 3158)       =     164.06
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4022
                                                Root MSE          =      .3989

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |   .0241362   .0203157     1.19   0.235     -.015697    .0639695
        prog |  -.0853152   .0422584    -2.02   0.044    -.1681719   -.0024584
         ks2 |   .0466904   .0037518    12.44   0.000     .0393342    .0540466
         fsm |  -.0114354   .0009439   -12.11   0.000    -.0132862   -.0095846
         eal


Teaching philosophy -- English (n=3170):
 beta_trad= 0.024 p=0.235
 beta_prog=-0.085 p=0.044

Linear regression                               Number of obs     =      3,170
                                                F(11, 3158)       =     182.56
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4226
                                                Root MSE          =     .36891



------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |   .0425094   .0196502     2.16   0.031      .003981    .0810378
        prog |  -.0848858   .0318375    -2.67   0.008    -.1473101   -.0224615
         ks2 |   .0302269    .003525     8.57   0.000     .0233154    .0371385
         fsm |   -.015603    .000822   -18.98   0.000    -.0172147   -.0139912
         eal |   .0106124    .000496    21.40   0.000     .0096399    .0115848
         sen |  -.0009365   .0011917    -0.79   0.432    -.0032731    .0014002
    log_size |   .0918512    .019325     4.75   0.000     .0539605     .129742
years_sinc~d |   .0083107   .0017578     4.73   0.000     .0048641    .0117573
     academy |  -.0169713   .0166812    -1.02   0.309    -.0496784    .0157358
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |   .0653107    .022104     2.95   0.003     .0219711    .1086504
        prog |  -.1129032   .0483445    -2.34   0.020    -.2076931   -.0181133
         ks2 |   .0491745   .0040144    12.25   0.000     .0413034    .0570456
         fsm |  -.0173582   .0010857   -15.99   0.000     -.019487   -.0152294
         eal |   .0125865   .0006793    18.53   0.000     .0112545    .0139185
         sen |  -.0010107   .0015427    -0.66   0.512    -.0040356    .0020142
    log_size |   .1181829   .0335758     3.52   0.000     .0523503    .1840155
years_sinc~d |   .0111379   .0021662     5.14   0.000     .0068907    .0153851
     academy |   .0235759   .0204309     1.15   0.249    -.0164833    .0636351
   urban_bin |


Linear regression                               Number of obs     =      3,170
                                                F(11, 3158)       =     144.13
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3585
                                                Root MSE          =     .45572

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |   .0205886   .0224044     0.92   0.358    -.0233401    .0645172
        prog |  -.1292229   .0512426    -2.52   0.012     -.229695   -.0287508
         ks2 |   .0535006   .0040369    13.25   0.000     .0455855    .0614158
         fsm |  -.0107224   .0010553   -10.16   0.000    -.0127914   -.0086533
         eal

   .0334168   .0208263     1.60   0.109    -.0074175    .0742512
   urban_bin |  -.0437532    .030185    -1.45   0.147    -.1029374    .0154309
   selective |   .0681353   .0353714     1.93   0.054     -.001218    .1374887
       _cons |  -6.884854   .4657207   -14.78   0.000    -7.797999   -5.971708
------------------------------------------------------------------------------

Teaching philosophy -- Open (n=3170):
 beta_trad= 0.021 p=0.358
 beta_prog=-0.129 p=0.012


(file C:/Users/damia/OneDrive/Documents/Schools
    Project/thesis/tables/tab_teaching_philosophy.tex not found)
(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_teaching_philosophy.tex)
tab_teaching_philosophy.tex written.
